In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import os
import sys
import json
import time
import random
import hashlib
import math
import gc
import shutil
import platform
import psutil

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import sklearn
import joblib
import torch
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

drive.mount('/content/drive')


# Initialize project paths

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

assert PROJECT_ROOT.is_dir()

CONFIG_DIR = PROJECT_ROOT / '00_config'
PROCESSED_DIR = PROJECT_ROOT / '01_data/processed'
SPLITS_DIR = PROJECT_ROOT / '01_data/splits'
FEATURES_DIR = PROJECT_ROOT / '01_data/features'
SRC_DIR = PROJECT_ROOT / '03_src'

NOTEBOOK_DIR = PROJECT_ROOT / '02_notebooks/06_model_optimization'
RESULTS_DIR = NOTEBOOK_DIR / 'results'

TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'

MODELS_DIR = PROJECT_ROOT / '04_models/optimization'

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'
DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

for directory in [
    TABLES_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR,
    PREDICTIONS_DIR,
    MODELS_DIR
]:

    directory.mkdir(parents=True, exist_ok=True)

print("Project directories initialized.")


# Define reusable utilities

def read_json(path):

    return json.loads(
        Path(path).read_text(encoding='utf-8')
    )


def save_json_atomic(path, content):

    path = Path(path)

    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_suffix(
        path.suffix + '.tmp'
    )

    temporary.write_text(
        json.dumps(content, indent=4, allow_nan=False),
        encoding='utf-8'
    )

    temporary.replace(path)


def save_csv_atomic(path, dataframe):

    path = Path(path)

    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_suffix(
        path.suffix + '.tmp'
    )

    dataframe.to_csv(temporary, index=False)

    temporary.replace(path)


def calculate_sha256(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as file:

        for chunk in iter(
            lambda: file.read(1024**2),
            b''
        ):

            digest.update(chunk)

    return digest.hexdigest()


def set_random_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Restore the original project configuration

config = read_json(
    CONFIG_DIR / 'project_config.json'
)

LABELS = config['labels']
NUM_LABELS = len(LABELS)
SEED = config['seed']

assert LABELS == [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

assert NUM_LABELS == 6
assert SEED == 42

set_random_seed(SEED)

print("Original project configuration verified.")


# Inspect the runtime environment

CUDA_AVAILABLE = torch.cuda.is_available()

DEVICE = torch.device(
    'cuda' if CUDA_AVAILABLE else 'cpu'
)

environment_info = {
    'Python': platform.python_version(),
    'NumPy': np.__version__,
    'Pandas': pd.__version__,
    'SciPy': scipy.__version__,
    'Scikit-learn': sklearn.__version__,
    'PyTorch': str(torch.__version__),
    'CPU cores': os.cpu_count(),
    'GPU available': CUDA_AVAILABLE,
    'GPU name': (
        torch.cuda.get_device_name(0)
        if CUDA_AVAILABLE else 'Not required'
    ),
    'Available RAM (GiB)':
        psutil.virtual_memory().available / 1024**3,
    'Local disk free (GiB)':
        psutil.disk_usage('/content').free / 1024**3
}

print("\nRuntime environment")

display(pd.DataFrame(
    list(environment_info.items()),
    columns=['Parameter', 'Value']
))


# Verify completion of previous notebooks

CLASSICAL_RESULTS_DIR = (
    PROJECT_ROOT
    / '02_notebooks/03_classical_ml'
    / '06_classical_ml_comparison/results'
)

DISTILBERT_RESULTS_DIR = (
    PROJECT_ROOT / '02_notebooks/04_distilbert/results'
)

ROBERTA_RESULTS_DIR = (
    PROJECT_ROOT / '02_notebooks/05_roberta/results'
)

CLASSICAL_SELECTION_PATH = (
    CLASSICAL_RESULTS_DIR
    / 'configs/classical_final_selection.json'
)

DISTILBERT_SELECTION_PATH = (
    DISTILBERT_RESULTS_DIR
    / 'configs/distilbert_final_selection.json'
)

ROBERTA_SELECTION_PATH = (
    ROBERTA_RESULTS_DIR
    / 'configs/roberta_final_selection.json'
)

DISTILBERT_COMPLETION_PATH = (
    DISTILBERT_RESULTS_DIR
    / 'logs/distilbert_completion_summary.json'
)

ROBERTA_COMPLETION_PATH = (
    ROBERTA_RESULTS_DIR
    / 'logs/roberta_completion_summary.json'
)

assert ROBERTA_COMPLETION_PATH.is_file(), (
    "Notebook 05 completion record is missing. "
    "Complete Cell 11 of Notebook 05 first."
)

classical_selection = read_json(
    CLASSICAL_SELECTION_PATH
)

distilbert_selection = read_json(
    DISTILBERT_SELECTION_PATH
)

roberta_selection = read_json(
    ROBERTA_SELECTION_PATH
)

distilbert_completion = read_json(
    DISTILBERT_COMPLETION_PATH
)

roberta_completion = read_json(
    ROBERTA_COMPLETION_PATH
)

assert distilbert_completion['status'] == 'Completed'
assert roberta_completion['status'] == 'Completed'

assert roberta_completion[
    'retained_artifacts_verified'
]

assert roberta_completion[
    'metrics_reconstructed_from_predictions'
]

for selection in [
    classical_selection,
    distilbert_selection,
    roberta_selection
]:

    assert selection['labels'] == LABELS
    assert not selection['official_test_used']
    assert not selection['final_project_model_frozen']

print("Previous notebook completion records verified.")


# Restore the experiment registry and decision log

registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

decisions = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert registry['run_id'].is_unique
assert registry['status'].eq('Completed').all()

assert decisions['decision_id'].is_unique

assert set(
    f'D{index:03d}' for index in range(1, 69)
).issubset(decisions['decision_id'])

registry_original_hash = calculate_sha256(
    REGISTRY_PATH
)

decision_log_original_hash = calculate_sha256(
    DECISION_LOG_PATH
)

previous_registry = registry.loc[
    registry['notebook'].ne('06_model_optimization')
].copy()

assert len(previous_registry) == 17

for _, record in previous_registry.iterrows():

    assert (
        PROJECT_ROOT / record['model_path']
    ).exists(), (
        f"Missing model: {record['run_id']}"
    )

print("\nPrevious completed experiments:", len(previous_registry))
print("Total registered experiments:", len(registry))
print("Methodological decisions:", len(decisions))
print("Registered model availability: Verified")


# Restore the leading model candidates

PRIMARY_CLASSICAL_RUN = (
    classical_selection['primary_selected_run_id']
)

SECONDARY_CLASSICAL_RUN = (
    classical_selection['secondary_selected_run_id']
)

PRIMARY_DISTILBERT_RUN = (
    distilbert_selection['selected_run_id']
)

PRIMARY_ROBERTA_RUN = (
    roberta_selection['selected_run_id']
)

SECONDARY_ROBERTA_RUN = (
    roberta_selection['runner_up_run_id']
)

candidate_ids = [
    PRIMARY_CLASSICAL_RUN,
    SECONDARY_CLASSICAL_RUN,
    PRIMARY_DISTILBERT_RUN,
    PRIMARY_ROBERTA_RUN,
    SECONDARY_ROBERTA_RUN
]

assert len(set(candidate_ids)) == 5

assert set(candidate_ids).issubset(
    previous_registry['run_id']
)

candidate_df = previous_registry.loc[
    previous_registry['run_id'].isin(candidate_ids),
    [
        'run_id',
        'model',
        'loss_function',
        'training_samples',
        'validation_macro_ap',
        'validation_macro_f1',
        'training_time_seconds'
    ]
].copy()

for column in [
    'training_samples',
    'validation_macro_ap',
    'validation_macro_f1',
    'training_time_seconds'
]:

    candidate_df[column] = pd.to_numeric(
        candidate_df[column],
        errors='raise'
    )

candidate_df = candidate_df.sort_values(
    'validation_macro_ap',
    ascending=False
).reset_index(drop=True)

PRIMARY_TRANSFORMER_RUN = str(
    candidate_df.loc[
        candidate_df['run_id'].isin([
            PRIMARY_DISTILBERT_RUN,
            PRIMARY_ROBERTA_RUN
        ])
    ].iloc[0]['run_id']
)

print("\nLeading validation candidates")

display(candidate_df.round(6))

print("Full-data classical candidate:", PRIMARY_CLASSICAL_RUN)
print("Full-data transformer candidate:", PRIMARY_TRANSFORMER_RUN)


# Verify selected candidate artifacts against existing manifests

def verify_selected_classical(candidate):

    dependencies = [
        ('model_path', 'model_sha256'),
        ('config_path', 'config_sha256'),
        ('threshold_path', 'threshold_sha256'),
        ('predictions_path', 'predictions_sha256')
    ]

    for path_key, hash_key in dependencies:

        path = PROJECT_ROOT / candidate[path_key]

        assert path.is_file()

        assert calculate_sha256(path) == (
            candidate[hash_key]
        )

    return len(dependencies)


classical_verified_artifacts = 0

for candidate in classical_selection[
    'selected_models'
].values():

    classical_verified_artifacts += (
        verify_selected_classical(candidate)
    )

print(
    "\nSelected classical artifacts verified:",
    classical_verified_artifacts
)


def verify_transformer_manifest(run_id):

    record = previous_registry.loc[
        previous_registry['run_id'].eq(run_id)
    ].iloc[0]

    run_dir = (
        PROJECT_ROOT / record['results_path']
    )

    manifest = pd.read_csv(
        run_dir / 'tables/artifact_manifest.csv'
    )

    assert len(manifest) == 20
    assert manifest['Path'].is_unique
    assert manifest['Run ID'].eq(run_id).all()

    for _, artifact in manifest.iterrows():

        path = PROJECT_ROOT / artifact['Path']

        assert path.is_file()

        assert path.stat().st_size == int(
            artifact['Size Bytes']
        )

        assert calculate_sha256(path) == (
            artifact['SHA256']
        )

    summary = read_json(
        run_dir / 'logs/run_summary.json'
    )

    assert not summary['official_test_used']

    return len(manifest)


transformer_verified_artifacts = 0

for run_id in [
    PRIMARY_DISTILBERT_RUN,
    PRIMARY_ROBERTA_RUN,
    SECONDARY_ROBERTA_RUN
]:

    count = verify_transformer_manifest(run_id)

    transformer_verified_artifacts += count

    print(run_id, ":", count, "artifacts verified")


# Verify the unchanged development datasets

dataset_manifest = pd.read_csv(
    SPLITS_DIR / 'prepared_dataset_manifest.csv'
)

assert dataset_manifest['Partition'].is_unique

development_partitions = [
    'train_full',
    'train_representative',
    'validation'
]

expected_rows = {
    'train_full': 127628,
    'train_representative': 59997,
    'validation': 31911
}

dataset_paths = {}
dataset_hashes = {}
dataset_verification = []
label_support = {'Label': LABELS}

for partition in development_partitions:

    record = dataset_manifest.loc[
        dataset_manifest['Partition'].eq(partition)
    ]

    assert len(record) == 1

    record = record.iloc[0]

    path = (
        PROCESSED_DIR / f'{partition}.parquet'
    )

    assert path.is_file()

    actual_hash = calculate_sha256(path)

    assert actual_hash == record['SHA256']

    rows = pq.ParquetFile(path).metadata.num_rows

    assert rows == expected_rows[partition]

    assert rows == int(record['Rows'])

    targets = pd.read_parquet(
        path,
        columns=LABELS
    )

    assert not targets.isna().any().any()

    assert targets.isin([0, 1]).all().all()

    positive_counts = (
        targets.sum().astype(int).tolist()
    )

    assert all(count > 0 for count in positive_counts)

    label_support[partition] = positive_counts

    dataset_paths[partition] = path
    dataset_hashes[partition] = actual_hash

    dataset_verification.append({
        'Partition': partition,
        'Rows': rows,
        'SHA256': actual_hash,
        'Verification': 'Passed'
    })

    del targets

    print(partition, ": Verified")


FULL_TRAIN_PATH = dataset_paths['train_full']

REPRESENTATIVE_TRAIN_PATH = (
    dataset_paths['train_representative']
)

VALIDATION_PATH = dataset_paths['validation']

FULL_TRAINING_SAMPLES = expected_rows['train_full']

REPRESENTATIVE_TRAINING_SAMPLES = (
    expected_rows['train_representative']
)

VALIDATION_SAMPLES = expected_rows['validation']

EXPECTED_FULL_TRAINING_HASH = dataset_hashes[
    'train_full'
]

EXPECTED_REPRESENTATIVE_HASH = dataset_hashes[
    'train_representative'
]

EXPECTED_VALIDATION_HASH = dataset_hashes[
    'validation'
]

for selection in [
    classical_selection,
    distilbert_selection,
    roberta_selection
]:

    assert selection['training_dataset_sha256'] == (
        EXPECTED_REPRESENTATIVE_HASH
    )

    assert selection['validation_dataset_sha256'] == (
        EXPECTED_VALIDATION_HASH
    )

print("\nDevelopment dataset verification")

display(pd.DataFrame(dataset_verification))

label_support_df = pd.DataFrame(label_support)

print("\nDevelopment label support")

display(label_support_df)


# Restore the shared evaluation module

if str(SRC_DIR) not in sys.path:

    sys.path.insert(0, str(SRC_DIR))

import evaluation_utils

from evaluation_utils import (
    evaluate_multilabel,
    optimize_thresholds,
    apply_thresholds
)

evaluation_hash = calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
)

for selection in [
    classical_selection,
    distilbert_selection,
    roberta_selection
]:

    assert evaluation_hash == (
        selection['evaluation_module_sha256']
    )

assert Path(evaluation_utils.__file__).resolve() == (
    SRC_DIR / 'evaluation_utils.py'
).resolve()

print("Shared evaluation module verified.")


# Estimate the full-data training budget

EFFECTIVE_BATCH_SIZE = 32
MAX_EPOCHS = 3
WARMUP_RATIO = 0.10

full_updates_per_epoch = math.ceil(
    FULL_TRAINING_SAMPLES / EFFECTIVE_BATCH_SIZE
)

full_total_updates = (
    full_updates_per_epoch * MAX_EPOCHS
)

full_warmup_steps = math.ceil(
    full_total_updates * WARMUP_RATIO
)

scaling_factor = (
    FULL_TRAINING_SAMPLES
    / REPRESENTATIVE_TRAINING_SAMPLES
)

reference_transformer = candidate_df.loc[
    candidate_df['run_id'].eq(
        PRIMARY_TRANSFORMER_RUN
    )
].iloc[0]

estimated_full_training_minutes = (
    float(
        reference_transformer[
            'training_time_seconds'
        ]
    ) / 60 * scaling_factor
)

scaling_summary = {
    'Representative training samples':
        REPRESENTATIVE_TRAINING_SAMPLES,

    'Full training samples':
        FULL_TRAINING_SAMPLES,

    'Additional training samples':
        FULL_TRAINING_SAMPLES
        - REPRESENTATIVE_TRAINING_SAMPLES,

    'Data scaling factor':
        scaling_factor,

    'Validation samples':
        VALIDATION_SAMPLES,

    'Full-data transformer updates per epoch':
        full_updates_per_epoch,

    'Full-data maximum optimizer updates':
        full_total_updates,

    'Full-data warmup updates':
        full_warmup_steps,

    'Estimated transformer training minutes':
        estimated_full_training_minutes
}

print("\nFull-data scaling budget")

display(pd.DataFrame(
    list(scaling_summary.items()),
    columns=['Parameter', 'Value']
))


# Define the proposed optimization experiments

S01_RUN_ID = 'S01_LR_COMBINED_BALANCED_FULL_S42'

S02_RUN_ID = 'S02_ROBERTA_FOCAL_FULL_S42'

optimization_plan = [
    {
        'experiment_id': 'S01',
        'run_id': S01_RUN_ID,
        'experiment': 'Full-data classical scaling',
        'reference_run': PRIMARY_CLASSICAL_RUN,
        'training_samples': FULL_TRAINING_SAMPLES,
        'seed': SEED,
        'priority': 'High',
        'status': 'Planned',
        'rationale': (
            'Measure scaling benefit for the strongest '
            'validation-selected classical configuration.'
        )
    },
    {
        'experiment_id': 'S02',
        'run_id': S02_RUN_ID,
        'experiment': 'Full-data transformer scaling',
        'reference_run': PRIMARY_TRANSFORMER_RUN,
        'training_samples': FULL_TRAINING_SAMPLES,
        'seed': SEED,
        'priority': 'High',
        'status': 'Pending resource assessment',
        'rationale': (
            'Evaluate additional training data for the '
            'leading transformer using matched architecture, '
            'loss and epoch budget.'
        )
    },
    {
        'experiment_id': 'R01',
        'experiment': 'Additional seed robustness',
        'status': 'Proposed deferral',
        'rationale': (
            'Multiple additional transformer runs would '
            'substantially increase computation.'
        )
    },
    {
        'experiment_id': 'A01',
        'experiment': 'Alternate sequence length',
        'status': 'Proposed deferral',
        'rationale': (
            'Existing token-length diagnostics are available. '
            'Additional training is resource dependent.'
        )
    },
    {
        'experiment_id': 'A02',
        'experiment': 'Rare-preserving sampling',
        'status': 'Proposed omission',
        'rationale': (
            'Optional experiment; prioritize full-data scaling '
            'and mandatory downstream evaluation.'
        )
    },
    {
        'experiment_id': 'A03',
        'experiment': 'Model ensemble',
        'status': 'Proposed deferral',
        'rationale': (
            'Prioritize individually reproducible models '
            'and essential evaluation deliverables.'
        )
    },
    {
        'experiment_id': 'TH01',
        'experiment': 'Threshold stability analysis',
        'status': 'Planned',
        'rationale': (
            'Investigate sensitivity of validation-derived '
            'thresholds, particularly for rare labels.'
        )
    },
    {
        'experiment_id': 'FR01',
        'experiment': 'Final model and threshold freeze',
        'status': 'Required',
        'rationale': (
            'Finalize model selection and evaluation roster '
            'before official test access.'
        )
    }
]

optimization_plan_df = pd.DataFrame(
    optimization_plan
)

print("\nProposed optimization plan")

display(optimization_plan_df[[
    'experiment_id',
    'experiment',
    'status',
    'rationale'
]])


# Save the provisional optimization configuration

OPTIMIZATION_PLAN_PATH = (
    RUN_CONFIGS_DIR / 'optimization_plan.json'
)

optimization_configuration = {
    'notebook': '06_model_optimization',

    'reference_plan':
        'NLP04_Final_Project_Execution_Plan.pdf, page 11',

    'labels': LABELS,
    'seed': SEED,

    'previous_completed_experiments':
        len(previous_registry),

    'primary_classical_candidate':
        PRIMARY_CLASSICAL_RUN,

    'secondary_classical_candidate':
        SECONDARY_CLASSICAL_RUN,

    'primary_distilbert_candidate':
        PRIMARY_DISTILBERT_RUN,

    'primary_roberta_candidate':
        PRIMARY_ROBERTA_RUN,

    'primary_transformer_candidate':
        PRIMARY_TRANSFORMER_RUN,

    'development_dataset_hashes':
        dataset_hashes,

    'development_dataset_rows':
        expected_rows,

    'evaluation_module_sha256':
        evaluation_hash,

    'primary_selection_metric':
        'macro_average_precision',

    'threshold_selection_metric':
        'per_label_f1',

    'full_data_scaling_from_scratch':
        True,

    'full_data_scaling_status':
        'Not started',

    'transformer_scaling_interpretation':
        'Combined data and compute scaling effect',

    'planned_experiments':
        optimization_plan,

    'estimated_full_transformer_training_minutes':
        estimated_full_training_minutes,

    'resource_driven_revisions_finalized':
        False,

    'official_test_used':
        False,

    'final_project_model_frozen':
        False
}

if OPTIMIZATION_PLAN_PATH.exists():

    saved_plan = read_json(
        OPTIMIZATION_PLAN_PATH
    )

    for key in [
        'labels',
        'seed',
        'development_dataset_hashes',
        'primary_classical_candidate',
        'primary_transformer_candidate',
        'evaluation_module_sha256'
    ]:

        assert saved_plan[key] == (
            optimization_configuration[key]
        )

    print("\nExisting optimization plan preserved.")

else:

    save_json_atomic(
        OPTIMIZATION_PLAN_PATH,
        optimization_configuration
    )

    print("\nProvisional optimization plan saved.")


# Verify that previous research records remain unchanged

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)


# Measure active project storage

project_size = sum(
    path.stat().st_size
    for path in PROJECT_ROOT.rglob('*')
    if path.is_file()
) / 1024**3


# Display initialization summary

print("\nModel optimization initialization summary")

print("Previous completed experiments:", len(previous_registry))
print("Methodological decisions:", len(decisions))

print("\nFull training samples:", FULL_TRAINING_SAMPLES)

print(
    "Representative training samples:",
    REPRESENTATIVE_TRAINING_SAMPLES
)

print("Validation samples:", VALIDATION_SAMPLES)

print(
    "Full-data scaling factor:",
    round(scaling_factor, 4)
)

print("\nPrimary classical candidate:", PRIMARY_CLASSICAL_RUN)

print(
    "Primary transformer candidate:",
    PRIMARY_TRANSFORMER_RUN
)

print(
    "Selected classical artifacts verified:",
    classical_verified_artifacts
)

print(
    "Selected transformer artifacts verified:",
    transformer_verified_artifacts
)

print(
    "\nActive project storage (GiB):",
    round(project_size, 4)
)

print("Previous research records: Preserved")
print("Development datasets: Verified")
print("Optimization plan: Saved")

print("\nFeature extraction performed: No")
print("Model training performed: No")
print("Additional figures generated: 0")
print("Official test dataset: Not loaded")
print("Final project model frozen: No")

gc.collect()

print("\nCell 1 completed successfully.")

In [ ]:
from sklearn.base import clone
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfVectorizer

import tempfile
import ctypes

# Define a memory cleanup utility

def release_unused_memory():

    gc.collect()

    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass


def show_available_memory():

    print(
        "Available RAM (GiB):",
        round(
            psutil.virtual_memory().available / 1024**3,
            3
        ),
        flush=True
    )


# Verify the original feature configuration

assert calculate_sha256(FULL_TRAIN_PATH) == (
    EXPECTED_FULL_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)

SOURCE_FEATURE_MANIFEST = (
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)

assert calculate_sha256(SOURCE_FEATURE_MANIFEST) == (
    classical_selection['feature_manifest_sha256']
)

source_feature_manifest = pd.read_csv(
    SOURCE_FEATURE_MANIFEST
)

SOURCE_VECTORIZER_DIR = (
    PROJECT_ROOT / '04_models/classical/vectorizers'
)

source_vectorizer_paths = {
    'word': (
        SOURCE_VECTORIZER_DIR
        / 'combined_word_tfidf.joblib'
    ),
    'character': (
        SOURCE_VECTORIZER_DIR
        / 'combined_character_tfidf.joblib'
    )
}

source_vectorizer_hashes = {}

for representation, path in source_vectorizer_paths.items():

    relative_path = str(
        path.relative_to(PROJECT_ROOT)
    )

    record = source_feature_manifest.loc[
        source_feature_manifest['Path'].eq(
            relative_path
        )
    ]

    assert len(record) == 1

    actual_hash = calculate_sha256(path)

    assert actual_hash == record.iloc[0]['SHA256']

    source_vectorizer_hashes[representation] = (
        actual_hash
    )

    print(
        f"{representation}: "
        "Original vectorizer fingerprint verified."
    )


# Define full-data feature paths

S01_FEATURE_DIR = (
    MODELS_DIR / S01_RUN_ID / 'vectorizers'
)

S01_FEATURE_MANIFEST_PATH = (
    S01_FEATURE_DIR / 'feature_manifest.json'
)

LOCAL_FEATURE_DIR = (
    Path('/content/nlp04_s01_features')
)

LOCAL_FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FULL_TRAIN_FEATURE_PATH = (
    LOCAL_FEATURE_DIR / 'train_full_combined.npz'
)

FULL_VAL_FEATURE_PATH = (
    LOCAL_FEATURE_DIR / 'validation_combined.npz'
)

LOCAL_FEATURE_MANIFEST_PATH = (
    LOCAL_FEATURE_DIR / 'local_feature_manifest.json'
)

FULL_FEATURE_BUDGET = 50000
WORD_FEATURE_BUDGET = 25000
CHAR_FEATURE_BUDGET = 25000


# Load the unchanged development partitions

train_full_df = pd.read_parquet(
    FULL_TRAIN_PATH
)

val_df = pd.read_parquet(
    VALIDATION_PATH
)

assert len(train_full_df) == FULL_TRAINING_SAMPLES
assert len(val_df) == VALIDATION_SAMPLES

required_columns = [
    'id',
    'comment_text',
    'group_id',
    *LABELS
]

for partition, dataframe in [
    ('train_full', train_full_df),
    ('validation', val_df)
]:

    assert set(required_columns).issubset(
        dataframe.columns
    )

    assert not dataframe[
        required_columns
    ].isna().any().any()

    assert dataframe['id'].is_unique

    assert dataframe[LABELS].isin(
        [0, 1]
    ).all().all()

    assert dataframe['comment_text'].map(
        lambda text: isinstance(text, str)
    ).all()

    assert dataframe[
        'comment_text'
    ].str.strip().ne('').all()

    row_manifest = pd.read_csv(
        SPLITS_DIR / f'{partition}_ids.csv',
        dtype={
            'id': str,
            'group_id': str
        }
    )

    np.testing.assert_array_equal(
        dataframe['id'].astype(str).to_numpy(),
        row_manifest['id'].to_numpy()
    )

    np.testing.assert_array_equal(
        dataframe['group_id'].astype(str).to_numpy(),
        row_manifest['group_id'].to_numpy()
    )

    np.testing.assert_array_equal(
        row_manifest['row_index'].to_numpy(),
        np.arange(len(dataframe))
    )

    print(
        partition,
        ": Dataset alignment verified."
    )


# Verify train-validation separation

assert not (
    set(train_full_df['id'])
    & set(val_df['id'])
)

assert not (
    set(train_full_df['group_id'])
    & set(val_df['group_id'])
)

def normalize_text_key(text):

    return ' '.join(
        text.lower().split()
    )


full_text_keys = set(
    train_full_df['comment_text'].map(
        normalize_text_key
    )
)

validation_text_keys = set(
    val_df['comment_text'].map(
        normalize_text_key
    )
)

assert not (
    full_text_keys & validation_text_keys
)

del full_text_keys, validation_text_keys

representative_ids = pd.read_parquet(
    REPRESENTATIVE_TRAIN_PATH,
    columns=['id']
)

assert set(
    representative_ids['id']
).issubset(
    train_full_df['id']
)

del representative_ids

print("Train-validation leakage checks: Passed")
print("Representative subset membership: Verified")


# Prepare texts, targets and row identifiers

full_train_texts = (
    train_full_df['comment_text'].tolist()
)

val_texts = (
    val_df['comment_text'].tolist()
)

full_train_ids = (
    train_full_df['id'].astype(str).to_numpy()
)

val_comment_ids = (
    val_df['id'].astype(str).to_numpy()
)

y_train_full = train_full_df[
    LABELS
].to_numpy(
    dtype=np.int8,
    copy=True
)

y_val = val_df[
    LABELS
].to_numpy(
    dtype=np.int8,
    copy=True
)

assert y_train_full.shape == (
    FULL_TRAINING_SAMPLES,
    NUM_LABELS
)

assert y_val.shape == (
    VALIDATION_SAMPLES,
    NUM_LABELS
)

np.testing.assert_array_equal(
    y_train_full.sum(axis=0),
    label_support_df['train_full'].to_numpy()
)

np.testing.assert_array_equal(
    y_val.sum(axis=0),
    label_support_df['validation'].to_numpy()
)

del train_full_df, val_df, row_manifest

release_unused_memory()

print("\nMemory before feature extraction")

show_available_memory()


# Define the full-data feature specification

feature_specification = {
    'run_id': S01_RUN_ID,
    'reference_run_id': PRIMARY_CLASSICAL_RUN,

    'training_dataset_sha256':
        EXPECTED_FULL_TRAINING_HASH,

    'validation_dataset_sha256':
        EXPECTED_VALIDATION_HASH,

    'source_vectorizer_hashes':
        source_vectorizer_hashes,

    'training_samples':
        FULL_TRAINING_SAMPLES,

    'validation_samples':
        VALIDATION_SAMPLES,

    'word_features': WORD_FEATURE_BUDGET,
    'character_features': CHAR_FEATURE_BUDGET,

    'combined_features': FULL_FEATURE_BUDGET,

    'feature_order': [
        'word',
        'character'
    ],

    'combined_normalization': 'l2',

    'matrix_format': 'csr',

    'matrix_dtype': 'float32',

    'scikit_learn_version':
        sklearn.__version__,

    'fit_partition': 'train_full',

    'official_test_used': False
}


# Check whether full-data vectorizers already exist

reuse_vectorizers = False

existing_feature_manifest = None

if S01_FEATURE_DIR.exists():

    assert S01_FEATURE_MANIFEST_PATH.is_file(), (
        "Incomplete full-data vectorizer directory. "
        "Inspect before continuing."
    )

    existing_feature_manifest = read_json(
        S01_FEATURE_MANIFEST_PATH
    )

    assert existing_feature_manifest[
        'specification'
    ] == feature_specification

    for representation, artifact in (
        existing_feature_manifest[
            'vectorizers'
        ].items()
    ):

        path = PROJECT_ROOT / artifact['path']

        assert path.is_file()

        assert calculate_sha256(path) == (
            artifact['sha256']
        )

    reuse_vectorizers = True

    print("\nVerified full-data vectorizers will be reused.")

else:

    print("\nFull-data vectorizers will be fitted from scratch.")


# Define sparse matrix verification

def verify_sparse_matrix(
    matrix,
    expected_rows,
    expected_features
):

    assert sp.isspmatrix_csr(matrix)

    assert matrix.shape == (
        expected_rows,
        expected_features
    )

    assert matrix.dtype == np.float32

    assert np.isfinite(matrix.data).all()

    assert (matrix.data >= 0).all()

    assert matrix.nnz > 0

    matrix.check_format(
        full_check=True
    )


def sparse_memory_mib(matrix):

    return (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    ) / 1024**2


# Fit one representation at a time

feature_records = []
vectorizer_records = {}

with tempfile.TemporaryDirectory(
    prefix='s01_feature_parts_',
    dir=LOCAL_FEATURE_DIR
) as temporary_directory:

    temporary_directory = Path(
        temporary_directory
    )

    for representation in [
        'word',
        'character'
    ]:

        print(
            f"\nPreparing {representation} TF-IDF...",
            flush=True
        )

        source_vectorizer = joblib.load(
            source_vectorizer_paths[
                representation
            ]
        )

        assert isinstance(
            source_vectorizer,
            TfidfVectorizer
        )

        source_parameters = (
            source_vectorizer.get_params(
                deep=False
            )
        )

        assert source_parameters[
            'max_features'
        ] == 25000

        assert source_parameters[
            'min_df'
        ] == 2

        assert source_parameters[
            'norm'
        ] == 'l2'

        assert source_parameters[
            'sublinear_tf'
        ]

        assert source_parameters[
            'use_idf'
        ]

        assert source_parameters[
            'smooth_idf'
        ]

        assert np.dtype(
            source_parameters['dtype']
        ) == np.dtype(np.float32)

        expected_ngram_range = (
            (1, 2)
            if representation == 'word'
            else (2, 5)
        )

        expected_analyzer = (
            'word'
            if representation == 'word'
            else 'char'
        )

        assert source_parameters[
            'ngram_range'
        ] == expected_ngram_range

        assert source_parameters[
            'analyzer'
        ] == expected_analyzer

        vectorizer_filename = (
            f'full_{representation}_tfidf.joblib'
        )

        local_vectorizer_path = (
            temporary_directory
            / vectorizer_filename
        )

        if reuse_vectorizers:

            saved_vectorizer_path = (
                PROJECT_ROOT
                / existing_feature_manifest[
                    'vectorizers'
                ][representation]['path']
            )

            vectorizer = joblib.load(
                saved_vectorizer_path
            )

            assert vectorizer.get_params(
                deep=False
            ) == source_parameters

            fit_seconds = 0.0

            start_time = time.perf_counter()

            train_matrix = (
                vectorizer.transform(
                    full_train_texts
                ).tocsr()
            )

            training_transform_seconds = (
                time.perf_counter() - start_time
            )

        else:

            # Clone settings without retaining the old vocabulary

            vectorizer = clone(
                source_vectorizer
            )

            assert not hasattr(
                vectorizer,
                'vocabulary_'
            )

            assert vectorizer.get_params(
                deep=False
            ) == source_parameters

            print(
                "Fitting on full training data only...",
                flush=True
            )

            start_time = time.perf_counter()

            train_matrix = (
                vectorizer.fit_transform(
                    full_train_texts
                ).tocsr()
            )

            fit_seconds = (
                time.perf_counter() - start_time
            )

            training_transform_seconds = None

        del source_vectorizer

        verify_sparse_matrix(
            train_matrix,
            FULL_TRAINING_SAMPLES,
            25000
        )

        assert len(
            vectorizer.vocabulary_
        ) == 25000

        assert np.isfinite(
            vectorizer.idf_
        ).all()

        train_memory = sparse_memory_mib(
            train_matrix
        )

        train_nnz = train_matrix.nnz

        # Save the temporary training features to local disk

        sp.save_npz(
            temporary_directory
            / f'{representation}_train.npz',

            train_matrix,

            compressed=False
        )

        del train_matrix

        release_unused_memory()

        show_available_memory()

        # Transform validation using the training-fitted vectorizer

        print(
            "Transforming unchanged validation data...",
            flush=True
        )

        start_time = time.perf_counter()

        validation_matrix = (
            vectorizer.transform(
                val_texts
            ).tocsr()
        )

        validation_transform_seconds = (
            time.perf_counter() - start_time
        )

        verify_sparse_matrix(
            validation_matrix,
            VALIDATION_SAMPLES,
            25000
        )

        val_memory = sparse_memory_mib(
            validation_matrix
        )

        val_nnz = validation_matrix.nnz

        sp.save_npz(
            temporary_directory
            / f'{representation}_validation.npz',

            validation_matrix,

            compressed=False
        )

        del validation_matrix

        # Preserve the full-training fitted vectorizer

        if reuse_vectorizers:

            shutil.copyfile(
                saved_vectorizer_path,
                local_vectorizer_path
            )

        else:

            joblib.dump(
                vectorizer,
                local_vectorizer_path,
                compress=3
            )

        vectorizer_hash = calculate_sha256(
            local_vectorizer_path
        )

        reloaded_vectorizer = joblib.load(
            local_vectorizer_path
        )

        assert (
            reloaded_vectorizer.vocabulary_
            == vectorizer.vocabulary_
        )

        np.testing.assert_array_equal(
            reloaded_vectorizer.idf_,
            vectorizer.idf_
        )

        assert reloaded_vectorizer.get_params(
            deep=False
        ) == source_parameters

        vectorizer_records[representation] = {
            'path': str(
                (
                    S01_FEATURE_DIR
                    / vectorizer_filename
                ).relative_to(PROJECT_ROOT)
            ),

            'sha256': vectorizer_hash,

            'size_bytes':
                local_vectorizer_path.stat().st_size,

            'features': len(
                vectorizer.vocabulary_
            ),

            'analyzer': expected_analyzer,

            'ngram_range':
                list(expected_ngram_range),

            'fit_partition': 'train_full',

            'reload_verified': True
        }

        feature_records.append({
            'Representation': representation,

            'Features': 25000,

            'Training Rows':
                FULL_TRAINING_SAMPLES,

            'Validation Rows':
                VALIDATION_SAMPLES,

            'Training Nonzero Values':
                int(train_nnz),

            'Validation Nonzero Values':
                int(val_nnz),

            'Training Matrix Memory (MiB)':
                float(train_memory),

            'Validation Matrix Memory (MiB)':
                float(val_memory),

            'Fit and Training Transform Time (s)':
                float(fit_seconds)
                if not reuse_vectorizers
                else None,

            'Training Transform Only Time (s)':
                training_transform_seconds,

            'Validation Transform Time (s)':
                float(validation_transform_seconds),

            'Vectorizer Reused':
                reuse_vectorizers,

            'Verification': 'Passed'
        })

        del vectorizer, reloaded_vectorizer

        release_unused_memory()

        print(
            representation,
            "TF-IDF preparation completed."
        )

        show_available_memory()


    # Combine word and character features

    combined_records = []

    for partition, expected_rows, output_path in [
        (
            'train',
            FULL_TRAINING_SAMPLES,
            FULL_TRAIN_FEATURE_PATH
        ),
        (
            'validation',
            VALIDATION_SAMPLES,
            FULL_VAL_FEATURE_PATH
        )
    ]:

        print(
            f"\nCombining {partition} features...",
            flush=True
        )

        word_matrix = sp.load_npz(
            temporary_directory
            / f'word_{partition}.npz'
        )

        char_matrix = sp.load_npz(
            temporary_directory
            / f'character_{partition}.npz'
        )

        verify_sparse_matrix(
            word_matrix,
            expected_rows,
            WORD_FEATURE_BUDGET
        )

        verify_sparse_matrix(
            char_matrix,
            expected_rows,
            CHAR_FEATURE_BUDGET
        )

        combined_matrix = sp.hstack(
            [
                word_matrix,
                char_matrix
            ],
            format='csr',
            dtype=np.float32
        )

        del word_matrix, char_matrix

        release_unused_memory()

        # Match the original combined feature normalization

        combined_matrix = normalize(
            combined_matrix,
            norm='l2',
            copy=False
        )

        verify_sparse_matrix(
            combined_matrix,
            expected_rows,
            FULL_FEATURE_BUDGET
        )

        # Verify row normalization in small batches

        zero_rows = 0

        for start in range(
            0,
            expected_rows,
            2048
        ):

            block = combined_matrix[
                start:start + 2048
            ]

            row_norms = np.sqrt(
                np.asarray(
                    block.multiply(block).sum(
                        axis=1
                    )
                ).ravel()
            )

            nonzero_rows = (
                block.getnnz(axis=1) > 0
            )

            np.testing.assert_allclose(
                row_norms[nonzero_rows],
                1.0,
                rtol=0,
                atol=1e-5
            )

            assert (
                row_norms[~nonzero_rows] == 0
            ).all()

            zero_rows += int(
                (~nonzero_rows).sum()
            )

        combined_record = {
            'Partition': partition,

            'Rows': expected_rows,

            'Features': FULL_FEATURE_BUDGET,

            'Nonzero Values':
                int(combined_matrix.nnz),

            'Density (%)': float(
                combined_matrix.nnz
                / (
                    expected_rows
                    * FULL_FEATURE_BUDGET
                )
                * 100
            ),

            'Zero Feature Rows': zero_rows,

            'Matrix Memory (MiB)':
                float(
                    sparse_memory_mib(
                        combined_matrix
                    )
                ),

            'Format': 'CSR',

            'Dtype': 'float32',

            'L2 Normalization Verified': True
        }

        # Save combined matrices locally

        temporary_output_path = (
            output_path.with_suffix('.tmp.npz')
        )

        sp.save_npz(
            temporary_output_path,
            combined_matrix,
            compressed=False
        )

        temporary_output_path.replace(
            output_path
        )

        del combined_matrix, block, row_norms

        release_unused_memory()

        # Independently reload the saved features

        reloaded_matrix = sp.load_npz(
            output_path
        )

        verify_sparse_matrix(
            reloaded_matrix,
            expected_rows,
            FULL_FEATURE_BUDGET
        )

        assert reloaded_matrix.nnz == (
            combined_record['Nonzero Values']
        )

        # Independently reconstruct selected rows

        rng = np.random.default_rng(SEED)

        verification_indices = np.unique(
            np.concatenate([
                [0, expected_rows - 1],

                rng.choice(
                    expected_rows,
                    size=32,
                    replace=False
                )
            ])
        )

        texts = (
            full_train_texts
            if partition == 'train'
            else val_texts
        )

        verification_texts = [
            texts[int(index)]
            for index in verification_indices
        ]

        word_vectorizer = joblib.load(
            temporary_directory
            / 'full_word_tfidf.joblib'
        )

        char_vectorizer = joblib.load(
            temporary_directory
            / 'full_character_tfidf.joblib'
        )

        reconstructed = normalize(
            sp.hstack(
                [
                    word_vectorizer.transform(
                        verification_texts
                    ),

                    char_vectorizer.transform(
                        verification_texts
                    )
                ],

                format='csr',
                dtype=np.float32
            ),

            norm='l2',
            copy=False
        )

        difference = (
            reloaded_matrix[
                verification_indices
            ] - reconstructed
        ).tocsr()

        maximum_difference = (
            float(
                np.abs(
                    difference.data
                ).max()
            )
            if difference.nnz
            else 0.0
        )

        assert maximum_difference <= 1e-6

        combined_record.update({
            'Local Path': str(output_path),

            'SHA256':
                calculate_sha256(output_path),

            'File Size (MiB)':
                output_path.stat().st_size
                / 1024**2,

            'Verification Samples':
                len(verification_indices),

            'Maximum Reconstruction Difference':
                maximum_difference,

            'Verification': 'Passed'
        })

        combined_records.append(
            combined_record
        )

        del reloaded_matrix, reconstructed
        del difference, word_vectorizer, char_vectorizer

        release_unused_memory()

        print(
            partition,
            ": Combined feature verification passed."
        )

        show_available_memory()


    # Save the persistent vectorizer manifest

    feature_manifest = {
        'specification': feature_specification,

        'vectorizers': vectorizer_records,

        'feature_preparation': feature_records,

        'combined_feature_verification':
            combined_records,

        'train_validation_alignment_verified':
            True,

        'train_validation_overlap': 0,

        'original_vectorizer_settings_matched':
            True,

        'validation_used_for_vectorizer_fitting':
            False,

        'combined_matrices_retained_locally':
            True,

        'official_test_used': False
    }

    if not reuse_vectorizers:

        staging_dir = S01_FEATURE_DIR.with_name(
            'vectorizers_staging'
        )

        assert not staging_dir.exists()

        staging_dir.mkdir(
            parents=True,
            exist_ok=False
        )

        for representation in [
            'word',
            'character'
        ]:

            filename = (
                f'full_{representation}_tfidf.joblib'
            )

            shutil.copyfile(
                temporary_directory / filename,
                staging_dir / filename
            )

            assert calculate_sha256(
                staging_dir / filename
            ) == vectorizer_records[
                representation
            ]['sha256']

        save_json_atomic(
            staging_dir / 'feature_manifest.json',
            feature_manifest
        )

        assert not S01_FEATURE_DIR.exists()

        staging_dir.rename(
            S01_FEATURE_DIR
        )

        print("\nFull-data vectorizers saved to Drive.")

    else:

        print(
            "\nExisting verified vectorizers preserved."
        )


# Save the current local matrix information

save_json_atomic(
    LOCAL_FEATURE_MANIFEST_PATH,
    feature_manifest
)


# Verify persistent vectorizers

persistent_feature_manifest = read_json(
    S01_FEATURE_MANIFEST_PATH
)

assert persistent_feature_manifest[
    'specification'
] == feature_specification

for representation, record in (
    persistent_feature_manifest[
        'vectorizers'
    ].items()
):

    path = PROJECT_ROOT / record['path']

    assert path.is_file()

    assert calculate_sha256(path) == (
        record['sha256']
    )

    assert record['sha256'] == (
        vectorizer_records[
            representation
        ]['sha256']
    )

FULL_WORD_VECTORIZER_PATH = (
    PROJECT_ROOT
    / vectorizer_records['word']['path']
)

FULL_CHAR_VECTORIZER_PATH = (
    PROJECT_ROOT
    / vectorizer_records['character']['path']
)

S01_FEATURE_MANIFEST_HASH = calculate_sha256(
    S01_FEATURE_MANIFEST_PATH
)


# Verify the original research inputs remain unchanged

assert calculate_sha256(FULL_TRAIN_PATH) == (
    EXPECTED_FULL_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)

for representation, path in (
    source_vectorizer_paths.items()
):

    assert calculate_sha256(path) == (
        source_vectorizer_hashes[
            representation
        ]
    )


# Display feature extraction results

feature_preparation_df = pd.DataFrame(
    feature_records
)

combined_feature_df = pd.DataFrame(
    combined_records
)

print("\nIndividual feature preparation")

display(
    feature_preparation_df.round(6)
)

print("\nCombined feature matrices")

display(
    combined_feature_df[[
        'Partition',
        'Rows',
        'Features',
        'Nonzero Values',
        'Density (%)',
        'Zero Feature Rows',
        'Matrix Memory (MiB)',
        'Maximum Reconstruction Difference',
        'Verification'
    ]].round(6)
)


# Release raw texts from memory

del full_train_texts, val_texts

release_unused_memory()


# Display final completion summary

print("\nFull-data feature preparation summary")

print("Training samples:", FULL_TRAINING_SAMPLES)

print("Validation samples:", VALIDATION_SAMPLES)

print("Word features:", WORD_FEATURE_BUDGET)

print("Character features:", CHAR_FEATURE_BUDGET)

print("Combined features:", FULL_FEATURE_BUDGET)

print("\nOriginal vectorizer settings: Matched")
print("Vectorizer fitting source: Full training only")
print("Validation feature transformation: Verified")
print("Combined feature normalization: Verified")
print("Feature reconstruction: Passed")

print("\nPersistent vectorizers: Saved and verified")
print("Combined sparse matrices: Saved locally")
print("Temporary feature components: Removed")
print("Original representative features: Preserved")

print(
    "\nLocal combined feature storage (MiB):",
    round(
        (
            FULL_TRAIN_FEATURE_PATH.stat().st_size
            + FULL_VAL_FEATURE_PATH.stat().st_size
        ) / 1024**2,
        3
    )
)

print("\nFinal memory availability")

show_available_memory()

print("\nExperiment registry: Unchanged")
print("Decision log: Unchanged")
print("Model training performed: No")
print("Additional figures generated: 0")
print("Official test dataset: Not loaded")

print("\nCell 2 completed successfully.")

In [ ]:
from sklearn.base import clone
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

import warnings


# Verify previous experimental records

current_registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert current_registry['run_id'].is_unique
assert len(current_registry) == 17

assert not current_registry['run_id'].eq(S01_RUN_ID).any(), (
    "S01 is already registered. Do not retrain."
)

registry_hash_before = calculate_sha256(REGISTRY_PATH)
decision_hash_before = calculate_sha256(DECISION_LOG_PATH)

assert calculate_sha256(FULL_TRAIN_PATH) == (
    EXPECTED_FULL_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(S01_FEATURE_MANIFEST_PATH) == (
    S01_FEATURE_MANIFEST_HASH
)


# Define experiment directories

S01_RESULTS_DIR = RESULTS_DIR / S01_RUN_ID

S01_TABLES_DIR = S01_RESULTS_DIR / 'tables'
S01_PREDICTIONS_DIR = S01_RESULTS_DIR / 'predictions'
S01_LOGS_DIR = S01_RESULTS_DIR / 'logs'

S01_MODEL_DIR = MODELS_DIR / S01_RUN_ID

for folder in [
    S01_TABLES_DIR,
    S01_PREDICTIONS_DIR,
    S01_LOGS_DIR,
    S01_MODEL_DIR
]:

    folder.mkdir(parents=True, exist_ok=True)

S01_MODEL_PATH = S01_MODEL_DIR / f'{S01_RUN_ID}.joblib'

S01_CONFIG_PATH = S01_RESULTS_DIR / 'config.json'

S01_THRESHOLD_PATH = S01_RESULTS_DIR / 'thresholds.json'

S01_PREDICTIONS_PATH = (
    S01_PREDICTIONS_DIR / 'validation_predictions.parquet'
)

S01_SUMMARY_PATH = (
    S01_LOGS_DIR / 'run_summary.json'
)

assert not S01_MODEL_PATH.exists(), (
    "An existing S01 model was found. Inspect before retraining."
)


# Restore the selected representative-subset model

reference_candidate = next(
    candidate
    for candidate in classical_selection['selected_models'].values()
    if candidate['run_id'] == PRIMARY_CLASSICAL_RUN
)

reference_model_path = (
    PROJECT_ROOT / reference_candidate['model_path']
)

assert calculate_sha256(reference_model_path) == (
    reference_candidate['model_sha256']
)

reference_model = joblib.load(reference_model_path)

assert isinstance(
    reference_model,
    OneVsRestClassifier
)

assert isinstance(
    reference_model.estimator,
    LogisticRegression
)

reference_parameters = (
    reference_model.estimator.get_params(deep=False)
)

assert reference_parameters['solver'] == 'liblinear'
assert reference_parameters['penalty'] == 'l2'
assert reference_parameters['C'] == 1.0
assert reference_parameters['class_weight'] == 'balanced'
assert reference_parameters['random_state'] == SEED
assert reference_model.n_jobs == 1


# Clone the model without retaining fitted coefficients

full_lr_model = clone(reference_model)

assert type(full_lr_model) is type(reference_model)

assert type(full_lr_model.estimator) is type(
    reference_model.estimator
)

# Compare estimator hyperparameters rather than object identity

assert full_lr_model.estimator.get_params(deep=False) == (
    reference_model.estimator.get_params(deep=False)
)

reference_wrapper_parameters = {
    key: value
    for key, value in reference_model.get_params(deep=False).items()
    if key != 'estimator'
}

cloned_wrapper_parameters = {
    key: value
    for key, value in full_lr_model.get_params(deep=False).items()
    if key != 'estimator'
}

assert cloned_wrapper_parameters == (
    reference_wrapper_parameters
)

assert not hasattr(full_lr_model, 'estimators_')
assert not hasattr(full_lr_model.estimator, 'coef_')

print("Original Logistic Regression configuration matched.")
print("One-vs-Rest configuration matched.")
print("Unfitted model clone verified.")

display(pd.DataFrame(
    [
        (key, str(value))
        for key, value in reference_parameters.items()
    ],
    columns=['Parameter', 'Value']
))

del reference_model

release_unused_memory()


# Verify locally stored feature matrices

local_feature_manifest = read_json(
    LOCAL_FEATURE_MANIFEST_PATH
)

assert local_feature_manifest['specification'] == (
    feature_specification
)

combined_feature_records = {
    item['Partition']: item
    for item in local_feature_manifest[
        'combined_feature_verification'
    ]
}

for partition, path in [
    ('train', FULL_TRAIN_FEATURE_PATH),
    ('validation', FULL_VAL_FEATURE_PATH)
]:

    assert path.is_file(), (
        "Local features unavailable. Rerun Cell 2."
    )

    assert calculate_sha256(path) == (
        combined_feature_records[partition]['SHA256']
    )

    print(partition, ": Feature fingerprint verified.")


# Verify the persistent vectorizers

persistent_feature_manifest = read_json(
    S01_FEATURE_MANIFEST_PATH
)

for representation, record in (
    persistent_feature_manifest['vectorizers'].items()
):

    path = PROJECT_ROOT / record['path']

    assert path.is_file()

    assert calculate_sha256(path) == record['sha256']

print("Persistent full-data vectorizers verified.")


# Verify training targets and row alignment

full_targets_df = pd.read_parquet(
    FULL_TRAIN_PATH,
    columns=['id', *LABELS]
)

validation_targets_df = pd.read_parquet(
    VALIDATION_PATH,
    columns=['id', *LABELS]
)

np.testing.assert_array_equal(
    full_targets_df['id'].astype(str).to_numpy(),
    full_train_ids
)

np.testing.assert_array_equal(
    validation_targets_df['id'].astype(str).to_numpy(),
    val_comment_ids
)

np.testing.assert_array_equal(
    full_targets_df[LABELS].to_numpy(dtype=np.int8),
    y_train_full
)

np.testing.assert_array_equal(
    validation_targets_df[LABELS].to_numpy(dtype=np.int8),
    y_val
)

del full_targets_df, validation_targets_df

print("Training and validation target alignment: Verified.")


# Calculate full-training balanced class weights

weight_records = []

for index, label in enumerate(LABELS):

    targets = y_train_full[:, index]

    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1]),
        y=targets
    )

    positive_count = int(targets.sum())

    negative_count = (
        FULL_TRAINING_SAMPLES - positive_count
    )

    weight_records.append({
        'Label': label,
        'Positive Samples': positive_count,
        'Negative Samples': negative_count,
        'Negative Class Weight': float(weights[0]),
        'Positive Class Weight': float(weights[1])
    })

class_weights_df = pd.DataFrame(weight_records)

print("\nFull-training balanced class weights")

display(class_weights_df.round(6))


# Load the full training feature matrix

print("\nLoading full training features...", flush=True)

X_train_full = sp.load_npz(
    FULL_TRAIN_FEATURE_PATH
)

verify_sparse_matrix(
    X_train_full,
    FULL_TRAINING_SAMPLES,
    FULL_FEATURE_BUDGET
)

assert X_train_full.shape[0] == len(y_train_full)

print("Training matrix shape:", X_train_full.shape)

print(
    "Training matrix memory (MiB):",
    round(sparse_memory_mib(X_train_full), 3)
)

show_available_memory()

assert psutil.virtual_memory().available / 1024**3 >= 4, (
    "Insufficient available RAM for safe model fitting."
)


# Train the full-data Logistic Regression model

set_random_seed(SEED)

start_timestamp = datetime.now(
    timezone.utc
).isoformat()

print(
    "\nTraining full-data balanced Logistic Regression...",
    flush=True
)

training_start = time.perf_counter()

with warnings.catch_warnings(record=True) as caught_warnings:

    warnings.simplefilter('always')

    with threadpool_limits(limits=2):

        full_lr_model.fit(
            X_train_full,
            y_train_full
        )

training_seconds = (
    time.perf_counter() - training_start
)

training_warning_records = [
    {
        'Category': item.category.__name__,
        'Message': str(item.message)
    }
    for item in caught_warnings
]

convergence_warning_count = sum(
    issubclass(item.category, ConvergenceWarning)
    for item in caught_warnings
)

print("\nTraining completed.")

print(
    "Training time (seconds):",
    round(training_seconds, 3)
)

print("Convergence warnings:", convergence_warning_count)

print("Binary classifiers:", len(full_lr_model.estimators_))

assert len(full_lr_model.estimators_) == NUM_LABELS
assert full_lr_model.multilabel_
assert full_lr_model.n_features_in_ == FULL_FEATURE_BUDGET


# Verify the fitted binary classifiers

estimator_records = []

for label, estimator in zip(
    LABELS,
    full_lr_model.estimators_
):

    assert isinstance(estimator, LogisticRegression)

    np.testing.assert_array_equal(
        estimator.classes_,
        np.array([0, 1])
    )

    assert estimator.coef_.shape == (
        1, FULL_FEATURE_BUDGET
    )

    assert np.isfinite(estimator.coef_).all()
    assert np.isfinite(estimator.intercept_).all()

    iterations = int(
        np.max(estimator.n_iter_)
    )

    estimator_records.append({
        'Label': label,
        'Features': estimator.n_features_in_,
        'Iterations': iterations,
        'Maximum Iterations': estimator.max_iter,
        'Nonzero Coefficients': int(
            np.count_nonzero(estimator.coef_)
        ),
        'Finite Coefficients': True
    })

estimator_verification_df = pd.DataFrame(
    estimator_records
)

print("\nFitted classifier verification")

display(estimator_verification_df)

save_json_atomic(
    S01_LOGS_DIR / 'training_warnings.json',
    training_warning_records
)

assert convergence_warning_count == 0, (
    "Convergence warning detected. Inspect before registration."
)


# Release the training matrix before validation

del X_train_full

release_unused_memory()

print("\nMemory after releasing training matrix")

show_available_memory()


# Load validation features

X_val_full = sp.load_npz(
    FULL_VAL_FEATURE_PATH
)

verify_sparse_matrix(
    X_val_full,
    VALIDATION_SAMPLES,
    FULL_FEATURE_BUDGET
)

assert X_val_full.shape[0] == len(y_val)

print("\nGenerating validation predictions...", flush=True)

prediction_start = time.perf_counter()

validation_scores = full_lr_model.predict_proba(
    X_val_full
)

prediction_seconds = (
    time.perf_counter() - prediction_start
)

assert validation_scores.shape == (
    VALIDATION_SAMPLES,
    NUM_LABELS
)

assert np.isfinite(validation_scores).all()

assert (
    (validation_scores >= 0)
    & (validation_scores <= 1)
).all()

print("Validation score shape:", validation_scores.shape)


# Evaluate default thresholds

default_metrics, default_per_label = (
    evaluate_multilabel(
        y_val,
        validation_scores,
        LABELS,
        thresholds=0.5
    )
)


# Optimize individual label thresholds on validation only

threshold_df = optimize_thresholds(
    y_val,
    validation_scores,
    LABELS
).set_index('Label').loc[LABELS].reset_index()

optimized_thresholds = threshold_df[
    'Optimal Threshold'
].to_numpy(dtype=np.float64)

assert optimized_thresholds.shape == (NUM_LABELS,)

assert np.isfinite(optimized_thresholds).all()

optimized_metrics, optimized_per_label = (
    evaluate_multilabel(
        y_val,
        validation_scores,
        LABELS,
        thresholds=optimized_thresholds
    )
)

assert np.isclose(
    default_metrics['Macro AP'],
    optimized_metrics['Macro AP'],
    rtol=0,
    atol=1e-12
)

assert optimized_metrics['Macro F1'] >= (
    default_metrics['Macro F1'] - 1e-12
)


# Save and independently reload the trained model

temporary_model_path = (
    S01_MODEL_PATH.with_suffix('.tmp.joblib')
)

joblib.dump(
    full_lr_model,
    temporary_model_path,
    compress=3
)

reloaded_model = joblib.load(
    temporary_model_path
)

assert isinstance(
    reloaded_model,
    OneVsRestClassifier
)

assert len(reloaded_model.estimators_) == NUM_LABELS

for original, reloaded in zip(
    full_lr_model.estimators_,
    reloaded_model.estimators_
):

    np.testing.assert_array_equal(
        original.coef_,
        reloaded.coef_
    )

    np.testing.assert_array_equal(
        original.intercept_,
        reloaded.intercept_
    )

reloaded_scores = reloaded_model.predict_proba(
    X_val_full
)

maximum_reload_error = float(
    np.max(np.abs(
        validation_scores - reloaded_scores
    ))
)

np.testing.assert_allclose(
    validation_scores,
    reloaded_scores,
    rtol=0,
    atol=1e-12
)

temporary_model_hash = calculate_sha256(
    temporary_model_path
)

temporary_model_path.replace(
    S01_MODEL_PATH
)

S01_MODEL_HASH = calculate_sha256(
    S01_MODEL_PATH
)

assert S01_MODEL_HASH == temporary_model_hash

print("\nModel reload verification")

print(
    "Maximum validation score difference:",
    maximum_reload_error
)

print("Model saved and verified.")

del full_lr_model, reloaded_model
del X_val_full, reloaded_scores

release_unused_memory()


# Prepare validation predictions

default_predictions = apply_thresholds(
    validation_scores,
    0.5
)

optimized_predictions = apply_thresholds(
    validation_scores,
    optimized_thresholds
)

predictions_df = pd.DataFrame({
    'id': val_comment_ids,
    'split': 'validation',
    'run_id': S01_RUN_ID
})

for index, label in enumerate(LABELS):

    predictions_df[f'true_{label}'] = (
        y_val[:, index]
    )

    predictions_df[f'score_{label}'] = (
        validation_scores[:, index]
    )

    predictions_df[f'pred_default_{label}'] = (
        default_predictions[:, index]
    )

    predictions_df[f'pred_optimized_{label}'] = (
        optimized_predictions[:, index]
    )

assert predictions_df['id'].is_unique

temporary_predictions_path = (
    S01_PREDICTIONS_PATH.with_suffix('.tmp.parquet')
)

predictions_df.to_parquet(
    temporary_predictions_path,
    index=False,
    compression='zstd'
)

pd.testing.assert_frame_equal(
    predictions_df,
    pd.read_parquet(temporary_predictions_path),
    check_exact=True
)

temporary_predictions_path.replace(
    S01_PREDICTIONS_PATH
)

pd.testing.assert_frame_equal(
    predictions_df,
    pd.read_parquet(S01_PREDICTIONS_PATH),
    check_exact=True
)


# Save the validation-derived thresholds

threshold_configuration = {
    'run_id': S01_RUN_ID,
    'labels': LABELS,
    'score_type': 'probability',
    'default_threshold': 0.5,
    'optimized_thresholds': {
        label: float(threshold)
        for label, threshold in zip(
            LABELS, optimized_thresholds
        )
    },
    'selection_partition': 'validation',
    'optimization_metric': 'per_label_f1',
    'comparison_operator': '>=',
    'tie_breaking': 'highest_threshold',
    'probability_calibration_established': False,
    'official_test_used': False
}

save_json_atomic(
    S01_THRESHOLD_PATH,
    threshold_configuration
)


# Compare default and optimized thresholds

overall_threshold_comparison = pd.DataFrame([
    {
        'Metric': metric,
        'Default': float(default_metrics[metric]),
        'Optimized': float(optimized_metrics[metric]),
        'Change': float(
            optimized_metrics[metric]
            - default_metrics[metric]
        )
    }
    for metric in default_metrics
])


# Compare the representative and full-data models

reference_registry_row = current_registry.loc[
    current_registry['run_id'].eq(PRIMARY_CLASSICAL_RUN)
].iloc[0]

reference_results_dir = (
    PROJECT_ROOT / reference_registry_row['results_path']
)

reference_per_label = pd.read_csv(
    reference_results_dir
    / 'tables/optimized_per_label_metrics.csv'
)

assert set(reference_per_label['Label']) == set(LABELS)

scaling_comparison_df = pd.DataFrame([
    {
        'Run ID': PRIMARY_CLASSICAL_RUN,
        'Training Samples': REPRESENTATIVE_TRAINING_SAMPLES,
        'Macro AP': float(
            reference_registry_row['validation_macro_ap']
        ),
        'Macro ROC-AUC': float(
            reference_candidate['validation_macro_roc_auc']
        ),
        'Optimized Macro F1': float(
            reference_registry_row['validation_macro_f1']
        ),
        'Optimized Micro F1': float(
            reference_candidate['validation_optimized_micro_f1']
        ),
        'Training Time (s)': float(
            reference_registry_row['training_time_seconds']
        )
    },
    {
        'Run ID': S01_RUN_ID,
        'Training Samples': FULL_TRAINING_SAMPLES,
        'Macro AP': float(
            optimized_metrics['Macro AP']
        ),
        'Macro ROC-AUC': float(
            optimized_metrics['Macro ROC-AUC']
        ),
        'Optimized Macro F1': float(
            optimized_metrics['Macro F1']
        ),
        'Optimized Micro F1': float(
            optimized_metrics['Micro F1']
        ),
        'Training Time (s)': float(
            training_seconds
        )
    }
])

comparison_columns = [
    'Label',
    'Average Precision',
    'Precision',
    'Recall',
    'F1',
    'TP',
    'FP',
    'FN'
]

scaling_per_label_df = reference_per_label[
    comparison_columns
].merge(
    optimized_per_label[comparison_columns],
    on='Label',
    suffixes=('_Representative', '_Full'),
    validate='one_to_one'
)

scaling_per_label_df['AP Change'] = (
    scaling_per_label_df['Average Precision_Full']
    - scaling_per_label_df['Average Precision_Representative']
)

scaling_per_label_df['F1 Change'] = (
    scaling_per_label_df['F1_Full']
    - scaling_per_label_df['F1_Representative']
)


# Save essential experimental tables

result_tables = {
    'class_weights.csv':
        class_weights_df,

    'estimator_verification.csv':
        estimator_verification_df,

    'default_overall_metrics.csv':
        pd.DataFrame([default_metrics]),

    'optimized_overall_metrics.csv':
        pd.DataFrame([optimized_metrics]),

    'default_per_label_metrics.csv':
        default_per_label,

    'optimized_per_label_metrics.csv':
        optimized_per_label,

    'optimized_thresholds.csv':
        threshold_df,

    'overall_threshold_comparison.csv':
        overall_threshold_comparison,

    'scaling_comparison.csv':
        scaling_comparison_df,

    'scaling_per_label_comparison.csv':
        scaling_per_label_df
}

for filename, dataframe in result_tables.items():

    save_csv_atomic(
        S01_TABLES_DIR / filename,
        dataframe
    )


# Prepare the experiment configuration

model_configuration = {
    'run_id': S01_RUN_ID,
    'reference_run_id': PRIMARY_CLASSICAL_RUN,
    'model': 'Logistic Regression',
    'classification_strategy': 'One-vs-Rest',
    'feature_representation': 'combined_tfidf',
    'sampling_strategy': 'full_training_partition',
    'class_weight': 'balanced',
    'seed': SEED,
    'labels': LABELS,

    'training_samples': FULL_TRAINING_SAMPLES,
    'validation_samples': VALIDATION_SAMPLES,

    'feature_budget': FULL_FEATURE_BUDGET,
    'word_features': WORD_FEATURE_BUDGET,
    'character_features': CHAR_FEATURE_BUDGET,
    'combined_normalization': 'l2',

    'estimator_parameters': reference_parameters,

    'wrapper_parameters':
        reference_wrapper_parameters,

    'original_model_configuration_matched': True,
    'trained_from_scratch': True,
    'vectorizers_refitted_on_full_training': True,

    'training_dataset_sha256':
        EXPECTED_FULL_TRAINING_HASH,

    'validation_dataset_sha256':
        EXPECTED_VALIDATION_HASH,

    'evaluation_module_sha256':
        evaluation_hash,

    'feature_manifest_path': str(
        S01_FEATURE_MANIFEST_PATH.relative_to(PROJECT_ROOT)
    ),

    'feature_manifest_sha256':
        S01_FEATURE_MANIFEST_HASH,

    'feature_dependencies':
        persistent_feature_manifest['vectorizers'],

    'model_path': str(
        S01_MODEL_PATH.relative_to(PROJECT_ROOT)
    ),

    'model_sha256':
        S01_MODEL_HASH,

    'threshold_path': str(
        S01_THRESHOLD_PATH.relative_to(PROJECT_ROOT)
    ),

    'predictions_path': str(
        S01_PREDICTIONS_PATH.relative_to(PROJECT_ROOT)
    ),

    'start_time': start_timestamp,
    'training_completed': True,
    'official_test_used': False,
    'final_project_model_frozen': False
}

save_json_atomic(
    S01_CONFIG_PATH,
    model_configuration
)


# Summarize the full-data scaling results

macro_ap_change = float(
    optimized_metrics['Macro AP']
    - float(reference_registry_row['validation_macro_ap'])
)

macro_f1_change = float(
    optimized_metrics['Macro F1']
    - float(reference_registry_row['validation_macro_f1'])
)

s01_summary = {
    'run_id': S01_RUN_ID,
    'reference_run_id': PRIMARY_CLASSICAL_RUN,
    'model': 'Logistic Regression',
    'training_samples': FULL_TRAINING_SAMPLES,
    'validation_samples': VALIDATION_SAMPLES,
    'seed': SEED,
    'feature_budget': FULL_FEATURE_BUDGET,

    'validation_macro_ap': float(
        optimized_metrics['Macro AP']
    ),

    'validation_macro_roc_auc': float(
        optimized_metrics['Macro ROC-AUC']
    ),

    'validation_default_macro_f1': float(
        default_metrics['Macro F1']
    ),

    'validation_optimized_macro_f1': float(
        optimized_metrics['Macro F1']
    ),

    'validation_default_micro_f1': float(
        default_metrics['Micro F1']
    ),

    'validation_optimized_micro_f1': float(
        optimized_metrics['Micro F1']
    ),

    'macro_ap_change_from_reference':
        macro_ap_change,

    'macro_f1_change_from_reference':
        macro_f1_change,

    'training_time_seconds':
        float(training_seconds),

    'validation_prediction_time_seconds':
        float(prediction_seconds),

    'training_time_excludes_feature_extraction':
        True,

    'convergence_warnings':
        convergence_warning_count,

    'maximum_probability_reload_error':
        maximum_reload_error,

    'model_path':
        model_configuration['model_path'],

    'model_sha256':
        S01_MODEL_HASH,

    'config_path': str(
        S01_CONFIG_PATH.relative_to(PROJECT_ROOT)
    ),

    'config_sha256':
        calculate_sha256(S01_CONFIG_PATH),

    'threshold_path':
        model_configuration['threshold_path'],

    'threshold_sha256':
        calculate_sha256(S01_THRESHOLD_PATH),

    'predictions_path':
        model_configuration['predictions_path'],

    'predictions_sha256':
        calculate_sha256(S01_PREDICTIONS_PATH),

    'feature_manifest_sha256':
        S01_FEATURE_MANIFEST_HASH,

    'statistical_significance_established': False,
    'official_test_used': False,
    'final_project_model_frozen': False
}

save_json_atomic(
    S01_SUMMARY_PATH,
    s01_summary
)

save_csv_atomic(
    S01_TABLES_DIR / 'run_summary.csv',
    pd.DataFrame([s01_summary])
)


# Verify all retained experimental artifacts

artifact_paths = [
    S01_MODEL_PATH,
    S01_CONFIG_PATH,
    S01_THRESHOLD_PATH,
    S01_PREDICTIONS_PATH,
    S01_SUMMARY_PATH,
    S01_FEATURE_MANIFEST_PATH,
    FULL_WORD_VECTORIZER_PATH,
    FULL_CHAR_VECTORIZER_PATH,
    S01_LOGS_DIR / 'training_warnings.json',
    S01_TABLES_DIR / 'run_summary.csv'
]

artifact_paths.extend([
    S01_TABLES_DIR / filename
    for filename in result_tables
])

artifact_records = []

for path in sorted(set(artifact_paths)):

    assert path.is_file()
    assert path.stat().st_size > 0

    artifact_records.append({
        'Run ID': S01_RUN_ID,
        'Path': str(
            path.relative_to(PROJECT_ROOT)
        ),
        'Size Bytes': path.stat().st_size,
        'SHA256': calculate_sha256(path),
        'Verification': 'Passed'
    })

artifact_manifest_df = pd.DataFrame(
    artifact_records
)

S01_ARTIFACT_MANIFEST_PATH = (
    S01_TABLES_DIR / 'artifact_manifest.csv'
)

save_csv_atomic(
    S01_ARTIFACT_MANIFEST_PATH,
    artifact_manifest_df
)

saved_manifest = pd.read_csv(
    S01_ARTIFACT_MANIFEST_PATH
)

assert saved_manifest['Path'].is_unique
assert saved_manifest['Run ID'].eq(S01_RUN_ID).all()

for _, artifact in saved_manifest.iterrows():

    path = PROJECT_ROOT / artifact['Path']

    assert path.stat().st_size == int(
        artifact['Size Bytes']
    )

    assert calculate_sha256(path) == (
        artifact['SHA256']
    )

print(
    "\nRetained artifacts verified:",
    len(saved_manifest)
)


# Verify unchanged research inputs

assert calculate_sha256(FULL_TRAIN_PATH) == (
    EXPECTED_FULL_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_hash_before
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_hash_before
)

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash

assert calculate_sha256(S01_FEATURE_MANIFEST_PATH) == (
    S01_FEATURE_MANIFEST_HASH
)


# Independently reconstruct the saved validation metrics

saved_predictions = pd.read_parquet(
    S01_PREDICTIONS_PATH
)

np.testing.assert_array_equal(
    saved_predictions['id'].astype(str).to_numpy(),
    val_comment_ids
)

saved_targets = saved_predictions[
    [f'true_{label}' for label in LABELS]
].to_numpy(dtype=np.int8)

saved_scores = saved_predictions[
    [f'score_{label}' for label in LABELS]
].to_numpy(dtype=np.float64)

np.testing.assert_array_equal(
    saved_targets,
    y_val
)

reconstructed_metrics, _ = evaluate_multilabel(
    saved_targets,
    saved_scores,
    LABELS,
    thresholds=optimized_thresholds
)

for metric in optimized_metrics:

    assert np.isclose(
        reconstructed_metrics[metric],
        optimized_metrics[metric],
        rtol=0,
        atol=1e-12
    ), f"Metric reconstruction failed: {metric}"

print("Saved validation metrics reconstructed successfully.")


# Register the completed experiment

registry_record = {
    'run_id': S01_RUN_ID,
    'notebook': '06_model_optimization',
    'model': 'Logistic Regression',
    'feature_representation': 'combined_tfidf',
    'loss_function': 'Binary logistic loss',
    'sampling_strategy': 'full_training_partition',
    'training_samples': FULL_TRAINING_SAMPLES,
    'seed': SEED,
    'max_length': '',

    'config_path': str(
        S01_CONFIG_PATH.relative_to(PROJECT_ROOT)
    ),

    'model_path': str(
        S01_MODEL_PATH.relative_to(PROJECT_ROOT)
    ),

    'results_path': str(
        S01_RESULTS_DIR.relative_to(PROJECT_ROOT)
    ),

    'validation_macro_ap':
        s01_summary['validation_macro_ap'],

    'validation_macro_f1':
        s01_summary['validation_optimized_macro_f1'],

    'training_time_seconds':
        training_seconds,

    'status': 'Completed',
    'start_time': start_timestamp,

    'end_time': datetime.now(
        timezone.utc
    ).isoformat(),

    'notes': (
        'Full-data scaling; cloned balanced LR configuration; '
        'vectorizers refitted on full training only; '
        'validation-based threshold optimization; '
        'official test not evaluated'
    )
}

assert set(registry_record) == set(
    current_registry.columns
)

updated_registry = pd.concat([
    current_registry,
    pd.DataFrame([registry_record])[
        current_registry.columns
    ]
], ignore_index=True)

assert updated_registry['run_id'].is_unique

save_csv_atomic(
    REGISTRY_PATH,
    updated_registry
)

persisted_registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert len(persisted_registry) == 18
assert persisted_registry['run_id'].is_unique

assert persisted_registry.loc[
    persisted_registry['run_id'].eq(S01_RUN_ID),
    'status'
].tolist() == ['Completed']


# Display the experimental results

print("\nDefault versus optimized thresholds")

display(
    overall_threshold_comparison.round(6)
)

print("\nFull-data optimized per-label metrics")

display(
    optimized_per_label.round(6)
)

print("\nSelected validation thresholds")

display(
    threshold_df.round(6)
)

print("\nRepresentative subset versus full-data scaling")

display(
    scaling_comparison_df.round(6)
)

print("\nPer-label scaling comparison")

display(
    scaling_per_label_df.round(6)
)


# Identify the leading classical candidate

classical_scaling_winner = (
    S01_RUN_ID
    if macro_ap_change > 0
    else PRIMARY_CLASSICAL_RUN
)


# Display the final completion summary

print("\nS01 full-data Logistic Regression summary")

print("Training samples:", FULL_TRAINING_SAMPLES)
print("Validation samples:", VALIDATION_SAMPLES)

print(
    "\nValidation Macro AP:",
    round(s01_summary['validation_macro_ap'], 6)
)

print(
    "Validation Macro ROC-AUC:",
    round(s01_summary['validation_macro_roc_auc'], 6)
)

print(
    "Default Macro F1:",
    round(s01_summary['validation_default_macro_f1'], 6)
)

print(
    "Optimized Macro F1:",
    round(s01_summary['validation_optimized_macro_f1'], 6)
)

print(
    "Optimized Micro F1:",
    round(s01_summary['validation_optimized_micro_f1'], 6)
)

print(
    "\nMacro AP change (percentage points):",
    round(macro_ap_change * 100, 6)
)

print(
    "Macro F1 change (percentage points):",
    round(macro_f1_change * 100, 6)
)

print(
    "Leading classical candidate:",
    classical_scaling_winner
)

print(
    "\nModel fitting time (minutes):",
    round(training_seconds / 60, 3)
)

print("Convergence warnings:", convergence_warning_count)
print("Model reload error:", maximum_reload_error)

print(
    "Retained artifacts:",
    len(saved_manifest)
)

print("\nModel configuration: Matched")
print("Training from scratch: Verified")
print("Saved metrics: Reconstructed")
print("Experiment registry: Updated")

print(
    "Total registered experiments:",
    len(persisted_registry)
)

print("Decision log: Unchanged")
print("Additional figures generated: 0")
print("Official test dataset: Not loaded")
print("Final project model frozen: No")

release_unused_memory()

print("\nFinal memory availability")

show_available_memory()

print("\nCell 3 completed successfully.")

In [ ]:
from sklearn.metrics import f1_score

# Verify the completed experiment registry

current_registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert len(current_registry) == 18
assert current_registry['run_id'].is_unique
assert current_registry['status'].eq('Completed').all()

registry_hash_before = calculate_sha256(REGISTRY_PATH)
decision_hash_before = calculate_sha256(DECISION_LOG_PATH)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash


# Define the threshold stability experiment

BOOTSTRAP_REPLICATES = 200
BOOTSTRAP_SEED = SEED

STABILITY_RUN_IDS = [
    S01_RUN_ID,
    'T03_FOCAL_S42',
    'T05_WBCE_S42',
    'T06_FOCAL_S42'
]

assert set(STABILITY_RUN_IDS).issubset(
    current_registry['run_id']
)

print("Threshold stability candidates")

display(
    current_registry.loc[
        current_registry['run_id'].isin(STABILITY_RUN_IDS),
        [
            'run_id',
            'model',
            'validation_macro_ap',
            'validation_macro_f1'
        ]
    ]
)


# Load validation identifiers, groups and targets

stability_val_df = pd.read_parquet(
    VALIDATION_PATH,
    columns=['id', 'group_id', *LABELS]
)

assert len(stability_val_df) == VALIDATION_SAMPLES
assert stability_val_df['id'].is_unique

assert not stability_val_df[
    ['id', 'group_id', *LABELS]
].isna().any().any()

stability_ids = (
    stability_val_df['id'].astype(str).to_numpy()
)

stability_targets = (
    stability_val_df[LABELS].to_numpy(dtype=np.int8)
)

assert np.isin(stability_targets, [0, 1]).all()

np.testing.assert_array_equal(
    stability_targets.sum(axis=0),
    label_support_df['validation'].to_numpy()
)

np.testing.assert_array_equal(
    stability_ids,
    val_comment_ids
)


# Encode validation groups for group-aware resampling

unique_groups, group_indices = np.unique(
    stability_val_df['group_id'].astype(str).to_numpy(),
    return_inverse=True
)

NUMBER_OF_GROUPS = len(unique_groups)

group_indices = group_indices.astype(np.int32)

assert NUMBER_OF_GROUPS == 31857

print("\nValidation comments:", VALIDATION_SAMPLES)
print("Unique validation groups:", NUMBER_OF_GROUPS)
print("Bootstrap replicates:", BOOTSTRAP_REPLICATES)


# Load and verify saved validation scores

stability_models = {}
stability_source_records = []

for run_id in STABILITY_RUN_IDS:

    record = current_registry.loc[
        current_registry['run_id'].eq(run_id)
    ].iloc[0]

    run_dir = PROJECT_ROOT / record['results_path']

    summary_path = run_dir / 'logs/run_summary.json'

    summary = read_json(summary_path)

    assert not summary['official_test_used']

    predictions_path = (
        PROJECT_ROOT / summary['predictions_path']
    )

    threshold_path = (
        PROJECT_ROOT / summary['threshold_path']
    )

    assert calculate_sha256(predictions_path) == (
        summary['predictions_sha256']
    )

    # Verify the threshold artifact against its run manifest

    manifest = pd.read_csv(
        run_dir / 'tables/artifact_manifest.csv'
    )

    threshold_relative_path = str(
        threshold_path.relative_to(PROJECT_ROOT)
    )

    threshold_record = manifest.loc[
        manifest['Path'].eq(threshold_relative_path)
    ]

    assert len(threshold_record) == 1

    threshold_hash = calculate_sha256(threshold_path)

    assert threshold_hash == (
        threshold_record.iloc[0]['SHA256']
    )

    threshold_config = read_json(threshold_path)

    assert threshold_config['labels'] == LABELS

    assert threshold_config['selection_partition'] == (
        'validation'
    )

    assert threshold_config['comparison_operator'] == '>='

    assert not threshold_config['official_test_used']

    predictions = pd.read_parquet(predictions_path)

    assert predictions['id'].is_unique

    np.testing.assert_array_equal(
        predictions['id'].astype(str).to_numpy(),
        stability_ids
    )

    targets = predictions[
        [f'true_{label}' for label in LABELS]
    ].to_numpy(dtype=np.int8)

    scores = predictions[
        [f'score_{label}' for label in LABELS]
    ].to_numpy(dtype=np.float64)

    np.testing.assert_array_equal(
        targets,
        stability_targets
    )

    assert scores.shape == (
        VALIDATION_SAMPLES, NUM_LABELS
    )

    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

    thresholds = np.array([
        threshold_config['optimized_thresholds'][label]
        for label in LABELS
    ], dtype=np.float64)

    # Reconstruct the original thresholds

    reconstructed = optimize_thresholds(
        targets,
        scores,
        LABELS
    ).set_index('Label').loc[LABELS]

    np.testing.assert_allclose(
        thresholds,
        reconstructed['Optimal Threshold'].to_numpy(),
        rtol=0,
        atol=1e-12
    )

    metrics, per_label = evaluate_multilabel(
        targets,
        scores,
        LABELS,
        thresholds=thresholds
    )

    assert np.isclose(
        metrics['Macro AP'],
        summary['validation_macro_ap'],
        rtol=0,
        atol=1e-10
    )

    assert np.isclose(
        metrics['Macro F1'],
        summary['validation_optimized_macro_f1'],
        rtol=0,
        atol=1e-10
    )

    stability_models[run_id] = {
        'model': record['model'],
        'scores': scores,
        'thresholds': thresholds,
        'original_per_label': per_label,
        'macro_ap': float(metrics['Macro AP'])
    }

    stability_source_records.append({
        'run_id': run_id,
        'predictions_path': summary['predictions_path'],
        'predictions_sha256': summary['predictions_sha256'],
        'threshold_path': summary['threshold_path'],
        'threshold_sha256': threshold_hash
    })

    print(run_id, ": Predictions and thresholds verified.")

    del predictions, targets


# Prepare score bins for efficient threshold optimization

prepared_scores = {}

for run_id, model_record in stability_models.items():

    prepared_scores[run_id] = []

    for label_index in range(NUM_LABELS):

        scores = model_record['scores'][:, label_index]

        unique_scores, inverse_indices = np.unique(
            scores,
            return_inverse=True
        )

        prepared_scores[run_id].append({
            'unique_scores': unique_scores,
            'inverse_indices': inverse_indices,
            'targets': stability_targets[
                :, label_index
            ].astype(np.float64)
        })


# Compute the F1-optimal threshold for weighted observations

def bootstrap_optimal_threshold(
    prepared,
    row_weights
):

    unique_scores = prepared['unique_scores']
    inverse_indices = prepared['inverse_indices']
    targets = prepared['targets']

    total_positives = float(
        np.dot(row_weights, targets)
    )

    total_negatives = float(
        row_weights.sum() - total_positives
    )

    if total_positives == 0 or total_negatives == 0:
        return None

    # Aggregate observations sharing the same score

    total_counts = np.bincount(
        inverse_indices,
        weights=row_weights,
        minlength=len(unique_scores)
    )

    positive_counts = np.bincount(
        inverse_indices,
        weights=row_weights * targets,
        minlength=len(unique_scores)
    )

    # Evaluate all observed score thresholds

    predicted_positives = np.cumsum(
        total_counts[::-1]
    )[::-1]

    true_positives = np.cumsum(
        positive_counts[::-1]
    )[::-1]

    denominator = (
        predicted_positives + total_positives
    )

    f1_values = np.divide(
        2.0 * true_positives,
        denominator,
        out=np.zeros_like(denominator),
        where=denominator > 0
    )

    # Exclude scores absent from the resampled dataset

    valid = total_counts > 0

    best_f1 = np.max(f1_values[valid])

    tied_indices = np.flatnonzero(
        valid
        & np.isclose(
            f1_values,
            best_f1,
            rtol=0,
            atol=1e-12
        )
    )

    # Match the original highest-threshold tie rule

    selected_index = tied_indices[-1]

    return {
        'threshold': float(
            unique_scores[selected_index]
        ),
        'f1': float(
            f1_values[selected_index]
        ),
        'positive_support': int(total_positives)
    }


# Verify the bootstrap optimizer on the original validation data

original_weights = np.ones(
    VALIDATION_SAMPLES,
    dtype=np.float64
)

optimizer_checks = []

for run_id in STABILITY_RUN_IDS:

    model_record = stability_models[run_id]

    original_per_label = (
        model_record['original_per_label']
        .set_index('Label')
    )

    for index, label in enumerate(LABELS):

        result = bootstrap_optimal_threshold(
            prepared_scores[run_id][index],
            original_weights
        )

        expected_threshold = (
            model_record['thresholds'][index]
        )

        expected_f1 = float(
            original_per_label.loc[label, 'F1']
        )

        assert np.isclose(
            result['threshold'],
            expected_threshold,
            rtol=0,
            atol=1e-12
        ), f"Threshold optimizer mismatch: {run_id}, {label}"

        assert np.isclose(
            result['f1'],
            expected_f1,
            rtol=0,
            atol=1e-12
        )

        optimizer_checks.append({
            'Run ID': run_id,
            'Label': label,
            'Verification': 'Passed'
        })

assert len(optimizer_checks) == 24

print("\nBootstrap optimizer verification: 24/24 passed")


# Run shared group-aware bootstrap resampling

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_records = []

bootstrap_start = time.perf_counter()

print("\nStarting threshold stability analysis...", flush=True)

for replicate in range(1, BOOTSTRAP_REPLICATES + 1):

    # Sample complete groups with replacement

    sampled_groups = rng.integers(
        0,
        NUMBER_OF_GROUPS,
        size=NUMBER_OF_GROUPS
    )

    group_weights = np.bincount(
        sampled_groups,
        minlength=NUMBER_OF_GROUPS
    )

    row_weights = group_weights[
        group_indices
    ].astype(np.float64)

    assert int(group_weights.sum()) == NUMBER_OF_GROUPS

    # Use the same bootstrap sample for every candidate

    for run_id in STABILITY_RUN_IDS:

        for index, label in enumerate(LABELS):

            result = bootstrap_optimal_threshold(
                prepared_scores[run_id][index],
                row_weights
            )

            if result is None:
                continue

            bootstrap_records.append({
                'Replicate': replicate,
                'Run ID': run_id,
                'Label': label,
                'Bootstrap Threshold':
                    result['threshold'],
                'Bootstrap Optimized F1':
                    result['f1'],
                'Bootstrap Positive Support':
                    result['positive_support'],
                'Bootstrap Rows':
                    int(row_weights.sum())
            })

    if replicate % 25 == 0:

        print(
            f"Completed {replicate}/{BOOTSTRAP_REPLICATES} "
            "bootstrap replicates.",
            flush=True
        )

bootstrap_seconds = (
    time.perf_counter() - bootstrap_start
)

bootstrap_df = pd.DataFrame(
    bootstrap_records
)

assert not bootstrap_df.empty

assert not bootstrap_df.duplicated([
    'Replicate',
    'Run ID',
    'Label'
]).any()


# Summarize threshold stability for each label

stability_records = []

for run_id in STABILITY_RUN_IDS:

    model_record = stability_models[run_id]

    for index, label in enumerate(LABELS):

        subset = bootstrap_df.loc[
            bootstrap_df['Run ID'].eq(run_id)
            & bootstrap_df['Label'].eq(label)
        ]

        thresholds = subset[
            'Bootstrap Threshold'
        ].to_numpy()

        assert len(thresholds) >= (
            BOOTSTRAP_REPLICATES * 0.95
        )

        lower, q25, median, q75, upper = (
            np.percentile(
                thresholds,
                [2.5, 25, 50, 75, 97.5]
            )
        )

        original_threshold = float(
            model_record['thresholds'][index]
        )

        stability_records.append({
            'Run ID': run_id,
            'Model': model_record['model'],
            'Label': label,
            'Validation Positive Support': int(
                stability_targets[:, index].sum()
            ),
            'Original Threshold': original_threshold,
            'Bootstrap Mean Threshold': float(
                thresholds.mean()
            ),
            'Bootstrap Median Threshold': float(median),
            'Bootstrap Standard Deviation': float(
                thresholds.std(ddof=1)
            ),
            'Bootstrap Lower Percentile': float(lower),
            'Bootstrap Upper Percentile': float(upper),
            'Percentile Interval Width': float(
                upper - lower
            ),
            'Interquartile Range': float(q75 - q25),
            'Median Shift': float(
                median - original_threshold
            ),
            'Valid Replicates': len(thresholds),
            'Missing Replicates':
                BOOTSTRAP_REPLICATES - len(thresholds)
        })

threshold_stability_df = pd.DataFrame(
    stability_records
)

assert len(threshold_stability_df) == 24


# Display the leading transformer's threshold stability

print("\nRoBERTa Focal threshold stability")

display(
    threshold_stability_df.loc[
        threshold_stability_df['Run ID'].eq(
            'T06_FOCAL_S42'
        )
    ].round(6)
)


# Compare rare-label threshold variability

rare_labels = [
    'severe_toxic',
    'threat',
    'identity_hate'
]

rare_threshold_stability_df = (
    threshold_stability_df.loc[
        threshold_stability_df['Label'].isin(
            rare_labels
        )
    ].copy()
)

print("\nRare-label threshold stability")

display(
    rare_threshold_stability_df[[
        'Run ID',
        'Label',
        'Validation Positive Support',
        'Original Threshold',
        'Bootstrap Median Threshold',
        'Bootstrap Lower Percentile',
        'Bootstrap Upper Percentile',
        'Percentile Interval Width',
        'Valid Replicates'
    ]].round(6)
)


# Identify the most variable thresholds

print("\nLargest threshold percentile interval widths")

display(
    threshold_stability_df.sort_values(
        'Percentile Interval Width',
        ascending=False
    )[[
        'Run ID',
        'Label',
        'Original Threshold',
        'Bootstrap Median Threshold',
        'Percentile Interval Width'
    ]].head(10).round(6)
)


# Save the essential threshold stability artifacts

STABILITY_TABLE_PATH = (
    TABLES_DIR / 'validation_threshold_stability.csv'
)

STABILITY_REPLICATES_PATH = (
    TABLES_DIR / 'threshold_bootstrap_replicates.csv'
)

STABILITY_CONFIG_PATH = (
    RUN_CONFIGS_DIR / 'threshold_stability_analysis.json'
)

save_csv_atomic(
    STABILITY_TABLE_PATH,
    threshold_stability_df
)

save_csv_atomic(
    STABILITY_REPLICATES_PATH,
    bootstrap_df
)

stability_configuration = {
    'analysis': 'Validation threshold stability',
    'run_ids': STABILITY_RUN_IDS,
    'labels': LABELS,
    'validation_samples': VALIDATION_SAMPLES,
    'validation_groups': NUMBER_OF_GROUPS,
    'validation_dataset_sha256':
        EXPECTED_VALIDATION_HASH,
    'evaluation_module_sha256':
        evaluation_hash,
    'bootstrap_replicates':
        BOOTSTRAP_REPLICATES,
    'bootstrap_seed': BOOTSTRAP_SEED,
    'resampling_unit': 'normalized_comment_group',
    'resampling_with_replacement': True,
    'shared_bootstrap_samples_across_models': True,
    'optimization_metric': 'per_label_f1',
    'comparison_operator': '>=',
    'tie_breaking': 'highest_threshold',
    'percentiles': [2.5, 25, 50, 75, 97.5],
    'optimizer_verification_checks': 24,
    'summary_records': len(threshold_stability_df),
    'bootstrap_records': len(bootstrap_df),
    'source_artifacts': stability_source_records,
    'bootstrap_time_seconds': float(
        bootstrap_seconds
    ),
    'original_thresholds_changed': False,
    'model_training_performed': False,
    'official_test_used': False,
    'probability_calibration_established': False,
    'statistical_superiority_established': False,
    'interpretation': (
        'Exploratory threshold variability conditional on '
        'fixed trained models and the observed validation '
        'partition. Does not quantify training-seed '
        'variability or independent test performance.'
    ),
    'artifacts': {
        'summary': {
            'path': str(
                STABILITY_TABLE_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            'sha256': calculate_sha256(
                STABILITY_TABLE_PATH
            )
        },
        'replicates': {
            'path': str(
                STABILITY_REPLICATES_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            'sha256': calculate_sha256(
                STABILITY_REPLICATES_PATH
            )
        }
    }
}

save_json_atomic(
    STABILITY_CONFIG_PATH,
    stability_configuration
)


# Verify saved outputs and unchanged research inputs

assert len(
    pd.read_csv(STABILITY_TABLE_PATH)
) == 24

assert len(
    pd.read_csv(STABILITY_REPLICATES_PATH)
) == len(bootstrap_df)

assert read_json(
    STABILITY_CONFIG_PATH
) == stability_configuration

assert calculate_sha256(REGISTRY_PATH) == (
    registry_hash_before
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_hash_before
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

for record in stability_source_records:

    assert calculate_sha256(
        PROJECT_ROOT / record['predictions_path']
    ) == record['predictions_sha256']

    assert calculate_sha256(
        PROJECT_ROOT / record['threshold_path']
    ) == record['threshold_sha256']


# Display completion summary

print("\nThreshold stability analysis summary")

print("Models analyzed:", len(STABILITY_RUN_IDS))
print("Labels per model:", NUM_LABELS)
print("Validation samples:", VALIDATION_SAMPLES)
print("Validation groups:", NUMBER_OF_GROUPS)

print("\nBootstrap replicates:", BOOTSTRAP_REPLICATES)
print("Optimizer verification: 24/24 passed")
print("Threshold summary records:", len(threshold_stability_df))
print("Bootstrap result records:", len(bootstrap_df))

print(
    "Missing bootstrap results:",
    BOOTSTRAP_REPLICATES
    * NUM_LABELS
    * len(STABILITY_RUN_IDS)
    - len(bootstrap_df)
)

print(
    "Bootstrap execution time (seconds):",
    round(bootstrap_seconds, 3)
)

print("\nGroup-aware resampling: Completed")
print("Original thresholds: Preserved")
print("Original validation predictions: Preserved")
print("Threshold stability tables: Saved")
print("Analysis configuration: Saved")

print("\nTotal registered experiments: 18")
print("Experiment registry: Unchanged")
print("Decision log: Unchanged")
print("Additional model training performed: No")
print("Additional figures generated: 0")
print("Official test dataset: Not loaded")
print("Final project model frozen: No")

release_unused_memory()

print("\nCell 4 completed successfully.")

In [ ]:
import time
import json
import os
import hashlib
import pyarrow as pa
import pyarrow.parquet as pq
import transformers
import tokenizers

from transformers import AutoTokenizer

assert torch.cuda.is_available(), "GPU runtime required for S02."

assert calculate_sha256(FULL_TRAIN_PATH) == (
    EXPECTED_FULL_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

registry_hash_before = calculate_sha256(REGISTRY_PATH)
decision_hash_before = calculate_sha256(DECISION_LOG_PATH)

current_registry = pd.read_csv(REGISTRY_PATH)

assert len(current_registry) == 18
assert S02_RUN_ID not in current_registry['run_id'].values

# Load the reference RoBERTa configuration

reference_row = current_registry.loc[
    current_registry['run_id'].eq('T06_FOCAL_S42')
].iloc[0]

reference_config_path = (
    PROJECT_ROOT / reference_row['config_path']
)

S02_REFERENCE_CONFIG = read_json(
    reference_config_path
)

S02_REFERENCE_SIGNATURE = (
    S02_REFERENCE_CONFIG['run_signature']
)

S02_CHECKPOINT = 'FacebookAI/roberta-base'

S02_REVISION = (
    'e2da8e2f811d1448a5b465c236feacd80ffbac7b'
)

S02_MAX_LENGTH = 256
S02_PADDING_MULTIPLE = 8
S02_TOKENIZATION_BATCH = 1024

S02_SHARED_INITIALIZATION_DIR = (
    PROJECT_ROOT
    / '04_models/roberta/shared_initialization_s42'
)

S02_TOKEN_CACHE_DIR = (
    PROJECT_ROOT / '01_data/tokenized/roberta'
)

assert S02_SHARED_INITIALIZATION_DIR.is_dir()
assert S02_TOKEN_CACHE_DIR.is_dir()

assert int(
    S02_REFERENCE_SIGNATURE['maximum_length']
) == S02_MAX_LENGTH

assert int(
    S02_REFERENCE_SIGNATURE['seed']
) == SEED

assert int(
    S02_REFERENCE_SIGNATURE['effective_batch_size']
) == 32

assert int(
    S02_REFERENCE_SIGNATURE['maximum_epochs']
) == 3

# Load the original shared tokenizer

s02_tokenizer = AutoTokenizer.from_pretrained(
    str(S02_SHARED_INITIALIZATION_DIR),
    local_files_only=True,
    use_fast=True
)

assert s02_tokenizer.is_fast
assert s02_tokenizer.vocab_size == 50265
assert s02_tokenizer.pad_token_id == 1
assert s02_tokenizer.bos_token_id == 0
assert s02_tokenizer.eos_token_id == 2
assert s02_tokenizer.truncation_side == 'right'
assert s02_tokenizer.padding_side == 'right'

print("Original RoBERTa tokenizer loaded.")
print("Tokenizer vocabulary:", s02_tokenizer.vocab_size)
print("Maximum sequence length:", S02_MAX_LENGTH)
print("Transformers version:", transformers.__version__)
print("Tokenizers version:", tokenizers.__version__)

# Locate existing caches using recorded fingerprints

def locate_reference_cache(partition, expected_hash):

    candidates = sorted(
        S02_TOKEN_CACHE_DIR.glob(
            f'{partition}*.parquet'
        )
    )

    matches = [
        path for path in candidates
        if calculate_sha256(path) == expected_hash
    ]

    assert len(matches) == 1, (
        f"Unable to uniquely verify {partition} cache."
    )

    return matches[0]


S02_REFERENCE_TRAIN_CACHE_PATH = locate_reference_cache(
    'train_representative',
    S02_REFERENCE_SIGNATURE[
        'training_token_cache_sha256'
    ]
)

S02_VALIDATION_CACHE_PATH = locate_reference_cache(
    'validation',
    S02_REFERENCE_SIGNATURE[
        'validation_token_cache_sha256'
    ]
)

print("\nRepresentative training cache verified.")
print("Existing validation cache verified.")

# Load the development partitions

required_columns = [
    'id',
    'comment_text',
    'group_id',
    *LABELS
]

s02_full_train = pd.read_parquet(
    FULL_TRAIN_PATH,
    columns=required_columns
)

s02_validation = pd.read_parquet(
    VALIDATION_PATH,
    columns=required_columns
)

s02_representative = pd.read_parquet(
    REPRESENTATIVE_TRAIN_PATH,
    columns=required_columns
)

assert len(s02_full_train) == 127628
assert len(s02_validation) == 31911
assert len(s02_representative) == 59997

assert s02_full_train['id'].is_unique
assert s02_validation['id'].is_unique

assert not set(
    s02_full_train['id']
).intersection(s02_validation['id'])

assert not set(
    s02_full_train['group_id']
).intersection(s02_validation['group_id'])

assert set(
    s02_representative['id']
).issubset(s02_full_train['id'])

for frame in [
    s02_full_train,
    s02_validation,
    s02_representative
]:

    assert frame['comment_text'].notna().all()

    assert np.isin(
        frame[LABELS].to_numpy(),
        [0, 1]
    ).all()

print("\nDevelopment partitions verified.")
print("Training-validation group overlap: 0")

# Verify tokenizer compatibility with historical caches

def verify_reference_tokenization(
    cache_path,
    source_frame,
    sample_size=256
):

    cache = pq.read_table(cache_path)

    assert cache.num_rows == len(source_frame)

    np.testing.assert_array_equal(
        np.asarray(
            cache['id'].to_pylist(),
            dtype=str
        ),
        source_frame['id'].astype(str).to_numpy()
    )

    np.testing.assert_array_equal(
        cache['row_index'].to_numpy(),
        np.arange(len(source_frame))
    )

    rng = np.random.default_rng(SEED)

    sample_indices = np.sort(
        rng.choice(
            len(source_frame),
            size=sample_size,
            replace=False
        )
    )

    sample_texts = (
        source_frame.iloc[sample_indices]
        ['comment_text']
        .tolist()
    )

    fresh_tokens = s02_tokenizer(
        sample_texts,
        max_length=S02_MAX_LENGTH,
        truncation=True,
        padding=False,
        add_special_tokens=True,
        return_attention_mask=False,
        return_token_type_ids=False
    )['input_ids']

    cached_tokens = (
        cache.take(
            pa.array(sample_indices)
        )['input_ids'].to_pylist()
    )

    assert fresh_tokens == cached_tokens

    return sample_size


representative_checks = (
    verify_reference_tokenization(
        S02_REFERENCE_TRAIN_CACHE_PATH,
        s02_representative
    )
)

validation_checks = (
    verify_reference_tokenization(
        S02_VALIDATION_CACHE_PATH,
        s02_validation
    )
)

print("\nHistorical tokenizer compatibility")
print("Representative samples verified:", representative_checks)
print("Validation samples verified:", validation_checks)
print("Tokenization mismatches: 0")

# Preserve the historical token-cache schema

reference_schema = pq.ParquetFile(
    S02_REFERENCE_TRAIN_CACHE_PATH
).schema_arrow

assert {
    'id',
    'row_index',
    'input_ids'
}.issubset(reference_schema.names)

supported_columns = {
    'id',
    'row_index',
    'input_ids',
    'labels',
    *LABELS
}

assert set(
    reference_schema.names
).issubset(supported_columns)

print("\nToken cache schema:")
print(reference_schema)

# Define the full-data cache specification

s02_cache_specification = {
    'schema_version': 1,
    'run_id': S02_RUN_ID,
    'partition': 'train_full',
    'dataset_sha256': EXPECTED_FULL_TRAINING_HASH,
    'checkpoint': S02_CHECKPOINT,
    'revision': S02_REVISION,
    'maximum_length': S02_MAX_LENGTH,
    'truncation': True,
    'truncation_side': 'right',
    'padding': False,
    'padding_multiple': S02_PADDING_MULTIPLE,
    'add_special_tokens': True,
    'rows': len(s02_full_train),
    'labels': LABELS,
    'transformers_version': transformers.__version__,
    'tokenizers_version': tokenizers.__version__,
    'pyarrow_version': pa.__version__,
    'reference_training_cache_sha256':
        calculate_sha256(S02_REFERENCE_TRAIN_CACHE_PATH),
    'reference_validation_cache_sha256':
        calculate_sha256(S02_VALIDATION_CACHE_PATH),
    'tokenizer_json_sha256': calculate_sha256(
        S02_SHARED_INITIALIZATION_DIR / 'tokenizer.json'
    )
}

s02_cache_key = hashlib.sha256(
    json.dumps(
        s02_cache_specification,
        sort_keys=True
    ).encode('utf-8')
).hexdigest()

S02_TRAIN_CACHE_PATH = (
    S02_TOKEN_CACHE_DIR
    / f'train_full_s02_{s02_cache_key[:16]}.parquet'
)

S02_TOKEN_CACHE_MANIFEST_PATH = (
    S02_TRAIN_CACHE_PATH.with_suffix('.json')
)

# Generate full-data tokens without permanent padding

if (
    S02_TRAIN_CACHE_PATH.exists()
    and S02_TOKEN_CACHE_MANIFEST_PATH.exists()
):

    existing_manifest = read_json(
        S02_TOKEN_CACHE_MANIFEST_PATH
    )

    assert existing_manifest['specification'] == (
        s02_cache_specification
    )

    assert calculate_sha256(S02_TRAIN_CACHE_PATH) == (
        existing_manifest['cache_sha256']
    )

    print("\nExisting full-data cache verified and reused.")

else:

    assert not S02_TRAIN_CACHE_PATH.exists(), (
        "Incomplete cache found. Inspect before continuing."
    )

    temporary_path = (
        S02_TRAIN_CACHE_PATH.with_suffix('.tmp')
    )

    tokenization_start = time.perf_counter()

    total_rows = len(s02_full_train)

    print("\nStarting full-data tokenization...", flush=True)

    with pq.ParquetWriter(
        temporary_path,
        reference_schema,
        compression='zstd'
    ) as writer:

        for start in range(
            0,
            total_rows,
            S02_TOKENIZATION_BATCH
        ):

            end = min(
                start + S02_TOKENIZATION_BATCH,
                total_rows
            )

            batch = s02_full_train.iloc[start:end]

            encoded = s02_tokenizer(
                batch['comment_text'].tolist(),
                max_length=S02_MAX_LENGTH,
                truncation=True,
                padding=False,
                add_special_tokens=True,
                return_attention_mask=False,
                return_token_type_ids=False
            )

            token_ids = encoded['input_ids']

            assert len(token_ids) == len(batch)

            assert all(
                2 <= len(ids) <= S02_MAX_LENGTH
                for ids in token_ids
            )

            batch_data = {
                'id': batch['id'].astype(str).tolist(),
                'row_index': list(range(start, end)),
                'input_ids': token_ids
            }

            if 'labels' in reference_schema.names:

                batch_data['labels'] = (
                    batch[LABELS].to_numpy().tolist()
                )

            for label in LABELS:

                if label in reference_schema.names:

                    batch_data[label] = (
                        batch[label].tolist()
                    )

            batch_table = pa.Table.from_pydict(
                batch_data,
                schema=reference_schema
            )

            writer.write_table(batch_table)

            if (
                end % (S02_TOKENIZATION_BATCH * 10) == 0
                or end == total_rows
            ):

                print(
                    f"Tokenized {end:,}/{total_rows:,} comments",
                    flush=True
                )

    assert pq.ParquetFile(
        temporary_path
    ).metadata.num_rows == total_rows

    os.replace(
        temporary_path,
        S02_TRAIN_CACHE_PATH
    )

    tokenization_seconds = (
        time.perf_counter() - tokenization_start
    )

    s02_cache_manifest = {
        'specification': s02_cache_specification,
        'cache_key': s02_cache_key,
        'cache_path': str(
            S02_TRAIN_CACHE_PATH.relative_to(PROJECT_ROOT)
        ),
        'cache_sha256': calculate_sha256(
            S02_TRAIN_CACHE_PATH
        ),
        'rows': total_rows,
        'size_bytes': S02_TRAIN_CACHE_PATH.stat().st_size,
        'tokenization_seconds': tokenization_seconds,
        'official_test_used': False
    }

    save_json_atomic(
        S02_TOKEN_CACHE_MANIFEST_PATH,
        s02_cache_manifest
    )

    print(
        "\nTokenization time:",
        round(tokenization_seconds, 3),
        "seconds"
    )

# Verify the completed full-data cache

s02_cache_manifest = read_json(
    S02_TOKEN_CACHE_MANIFEST_PATH
)

assert calculate_sha256(S02_TRAIN_CACHE_PATH) == (
    s02_cache_manifest['cache_sha256']
)

full_cache_checks = verify_reference_tokenization(
    S02_TRAIN_CACHE_PATH,
    s02_full_train
)

full_cache = pq.read_table(
    S02_TRAIN_CACHE_PATH,
    columns=['input_ids']
)

token_lengths = np.asarray(
    [
        len(ids)
        for ids in full_cache['input_ids'].to_pylist()
    ],
    dtype=np.int32
)

assert len(token_lengths) == 127628
assert token_lengths.min() >= 2
assert token_lengths.max() <= S02_MAX_LENGTH

tokenization_summary = pd.DataFrame({
    'Parameter': [
        'Training samples',
        'Validation samples',
        'Minimum token length',
        'Mean token length',
        'Median token length',
        '95th percentile token length',
        'Maximum token length',
        'Sequences reaching length limit',
        'Full-cache size (MiB)',
        'Fresh tokenization checks'
    ],
    'Value': [
        len(s02_full_train),
        len(s02_validation),
        token_lengths.min(),
        token_lengths.mean(),
        np.median(token_lengths),
        np.percentile(token_lengths, 95),
        token_lengths.max(),
        int((token_lengths == S02_MAX_LENGTH).sum()),
        S02_TRAIN_CACHE_PATH.stat().st_size / 1024**2,
        full_cache_checks
    ]
})

print("\nFull-data tokenization summary")
display(tokenization_summary.round(4))

# Save the S02 tokenization configuration

S02_TOKENIZATION_CONFIG_PATH = (
    RUN_CONFIGS_DIR
    / 's02_tokenization_configuration.json'
)

s02_tokenization_configuration = {
    'run_id': S02_RUN_ID,
    'reference_run_id': 'T06_FOCAL_S42',
    'training_samples': len(s02_full_train),
    'validation_samples': len(s02_validation),
    'training_dataset_sha256':
        EXPECTED_FULL_TRAINING_HASH,
    'validation_dataset_sha256':
        EXPECTED_VALIDATION_HASH,
    'training_token_cache_path': str(
        S02_TRAIN_CACHE_PATH.relative_to(PROJECT_ROOT)
    ),
    'training_token_cache_sha256':
        calculate_sha256(S02_TRAIN_CACHE_PATH),
    'validation_token_cache_path': str(
        S02_VALIDATION_CACHE_PATH.relative_to(PROJECT_ROOT)
    ),
    'validation_token_cache_sha256':
        calculate_sha256(S02_VALIDATION_CACHE_PATH),
    'cache_manifest_path': str(
        S02_TOKEN_CACHE_MANIFEST_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    'cache_manifest_sha256':
        calculate_sha256(S02_TOKEN_CACHE_MANIFEST_PATH),
    'reference_config_sha256':
        calculate_sha256(reference_config_path),
    'shared_initialization_path': str(
        S02_SHARED_INITIALIZATION_DIR.relative_to(
            PROJECT_ROOT
        )
    ),
    'maximum_length': S02_MAX_LENGTH,
    'padding_multiple': S02_PADDING_MULTIPLE,
    'historical_tokenizer_checks':
        representative_checks + validation_checks,
    'full_cache_tokenization_checks': full_cache_checks,
    'validation_cache_reused': True,
    'training_validation_overlap': 0,
    'official_test_used': False
}

save_json_atomic(
    S02_TOKENIZATION_CONFIG_PATH,
    s02_tokenization_configuration
)

save_csv_atomic(
    TABLES_DIR / 's02_tokenization_summary.csv',
    tokenization_summary
)

# Release temporary data objects

del full_cache
del s02_full_train
del s02_validation
del s02_representative
del token_lengths

gc.collect()

assert calculate_sha256(REGISTRY_PATH) == (
    registry_hash_before
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_hash_before
)

print("\nS02 preparation summary")
print("Training comments tokenized: 127628")
print("Validation comments: 31911")
print("Historical validation cache: Reused")
print("Tokenizer compatibility: Verified")
print("Full-data cache: Saved and verified")
print("Training-validation leakage: 0")
print("Model training performed: No")
print("Experiment registry: Unchanged")
print("Decision log: Unchanged")
print("Official test dataset: Not loaded")
print("Final project model frozen: No")

print("\nCell 5 completed successfully.")

In [ ]:
script_path = SRC_DIR / '06_optimization_cell6.py'
exec(compile(script_path.read_text(), str(script_path), 'exec'), globals())


In [ ]:
# Verify training readiness

assert torch.cuda.is_available()

assert s02_pipeline_configuration[
    'training_configuration_finalized'
]

assert s02_smoke_summary[
    'exact_checkpoint_restore_verified'
]

assert s02_smoke_summary[
    'numerical_continuation_verified'
]

assert TRAINING_SAMPLES == 127628
assert VALIDATION_SAMPLES == 31911

RUN_ID = S02_RUN_ID
LOSS_NAME = 'FOCAL'

run_signature = get_run_signature(RUN_ID)

assert run_signature == read_json(
    S02_PIPELINE_CONFIG_PATH
)['run_signature']

registry_hash_before = calculate_sha256(REGISTRY_PATH)
decision_hash_before = calculate_sha256(DECISION_LOG_PATH)

assert RUN_ID not in pd.read_csv(
    REGISTRY_PATH
)['run_id'].values


# Prepare experiment directories

S02_RESULTS_DIR = RESULTS_DIR / RUN_ID
S02_MODEL_DIR = MODELS_DIR / RUN_ID

S02_TABLES_DIR = S02_RESULTS_DIR / 'tables'
S02_LOGS_DIR = S02_RESULTS_DIR / 'logs'
S02_PREDICTIONS_DIR = S02_RESULTS_DIR / 'predictions'
S02_FIGURES_DIR = S02_RESULTS_DIR / 'figures'

for folder in [
    S02_RESULTS_DIR,
    S02_MODEL_DIR,
    S02_TABLES_DIR,
    S02_LOGS_DIR,
    S02_PREDICTIONS_DIR,
    S02_FIGURES_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

S02_LOCAL_CHECKPOINT = (
    Path('/content/nlp04_runtime_checkpoints')
    / RUN_ID
    / 'latest_checkpoint.pt'
)

S02_DRIVE_CHECKPOINT = (
    S02_MODEL_DIR / 'latest_checkpoint.pt'
)

S02_CONFIG_PATH = S02_RESULTS_DIR / 'config.json'

S02_PROGRESS_PATH = (
    S02_LOGS_DIR / 'training_progress.json'
)

S02_HISTORY_PATH = (
    S02_TABLES_DIR / 'epoch_history.csv'
)

S02_LOCAL_CHECKPOINT.parent.mkdir(
    parents=True,
    exist_ok=True
)


# Save the experiment configuration

run_configuration = {
    'run_id': RUN_ID,
    'reference_run_id': 'T06_FOCAL_S42',
    'model': 'RoBERTa',
    'loss': LOSS_NAME,
    'sampling_strategy': 'full_training_pool',
    'training_samples': TRAINING_SAMPLES,
    'validation_samples': VALIDATION_SAMPLES,
    'trained_from_scratch_from_shared_initialization': True,
    'run_signature': run_signature,
    'determinism_policy_source': policy_source,
    'historical_determinism_policy_confirmed': (
        policy_source != 'S02_explicit_reproducibility_policy'
    ),
    'local_checkpoint_interval_updates': CHECKPOINT_INTERVAL,
    'persistent_checkpoint_interval': 'epoch',
    'selection_metric': 'macro_average_precision',
    'threshold_selection_partition': 'validation',
    'official_test_used': False,
    'final_project_model_frozen': False
}

if S02_CONFIG_PATH.exists():

    assert read_json(S02_CONFIG_PATH) == run_configuration

else:

    save_json_atomic(
        S02_CONFIG_PATH,
        run_configuration
    )


# Initialize training and restore progress if available

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model, optimizer, scheduler, scaler = (
    create_training_components(
        total_steps=TOTAL_TRAINING_STEPS,
        warmup_steps=WARMUP_STEPS
    )
)

progress = new_training_progress()

resume_path = None

if S02_LOCAL_CHECKPOINT.is_file():

    resume_path = S02_LOCAL_CHECKPOINT

elif S02_DRIVE_CHECKPOINT.is_file():

    resume_path = S02_DRIVE_CHECKPOINT

if resume_path is not None:

    progress = load_resume_checkpoint(
        resume_path,
        model,
        optimizer,
        scheduler,
        scaler,
        run_signature
    )

    print("\nCheckpoint restored:", resume_path)
    print("Resuming epoch:", progress['epoch'])
    print("Resuming row:", progress['next_row'])
    print("Completed updates:", progress['attempted_updates'])

else:

    assert model_state_sha256(model) == INITIAL_STATE_HASH

    assert not S02_PROGRESS_PATH.exists(), (
        "Training progress exists but recovery checkpoint is missing."
    )

    print("\nStarting from original shared initialization.")


# Save local recovery checkpoints

def s02_checkpoint_callback(current_progress):

    save_resume_checkpoint(
        S02_LOCAL_CHECKPOINT,
        model,
        optimizer,
        scheduler,
        scaler,
        current_progress,
        run_signature
    )

    print(
        "Local checkpoint saved | Update:",
        current_progress['attempted_updates'],
        flush=True
    )


# Persist the latest verified recovery checkpoint

def persist_s02_recovery(current_progress):

    save_resume_checkpoint(
        S02_LOCAL_CHECKPOINT,
        model,
        optimizer,
        scheduler,
        scaler,
        current_progress,
        run_signature
    )

    temporary_path = (
        S02_DRIVE_CHECKPOINT.with_suffix('.pt.tmp')
    )

    shutil.copyfile(
        S02_LOCAL_CHECKPOINT,
        temporary_path
    )

    assert temporary_path.stat().st_size == (
        S02_LOCAL_CHECKPOINT.stat().st_size
    )

    assert calculate_sha256(temporary_path) == (
        calculate_sha256(S02_LOCAL_CHECKPOINT)
    )

    temporary_path.replace(S02_DRIVE_CHECKPOINT)

    save_json_atomic(
        S02_PROGRESS_PATH,
        current_progress
    )

    save_csv_atomic(
        S02_HISTORY_PATH,
        pd.DataFrame(current_progress['history'])
    )

    print("Persistent recovery checkpoint saved and verified.")


# Display the training configuration

print("\nStarting S02 full-data RoBERTa experiment")

print("Training samples:", TRAINING_SAMPLES)
print("Validation samples:", VALIDATION_SAMPLES)
print("Loss function:", LOSS_NAME)
print("Maximum epochs:", MAX_EPOCHS)
print("Updates per epoch:", UPDATES_PER_EPOCH)
print("Maximum optimizer updates:", TOTAL_TRAINING_STEPS)
print("Warmup updates:", WARMUP_STEPS)
print("GPU:", torch.cuda.get_device_name(0))

session_start = time.perf_counter()


# Execute the full training experiment

while (
    not progress['training_finished']
    and progress['epoch'] <= MAX_EPOCHS
):

    epoch = progress['epoch']

    print(f"\nStarting epoch {epoch}/{MAX_EPOCHS}")
    print("-" * 65, flush=True)

    training_loss = train_one_epoch(
        model,
        optimizer,
        scheduler,
        scaler,
        LOSS_NAME,
        progress,
        checkpoint_callback=s02_checkpoint_callback
    )

    assert training_loss is not None
    assert np.isfinite(training_loss)

    assert progress['epoch_examples'] == TRAINING_SAMPLES

    assert progress['attempted_updates'] == (
        epoch * UPDATES_PER_EPOCH
    )

    print("\nRunning validation inference...", flush=True)

    validation_result = evaluate_validation_model(
        model,
        LOSS_NAME
    )

    metrics = validation_result['metrics']

    macro_ap = float(metrics['Macro AP'])
    macro_f1 = float(metrics['Macro F1'])
    micro_f1 = float(metrics['Micro F1'])
    macro_roc = float(metrics['Macro ROC-AUC'])

    assert np.isfinite(macro_ap)

    previous_best_path = progress['best_checkpoint_path']
    previous_best_predictions = progress['best_validation_path']

    improved = (
        progress['best_macro_ap'] is None
        or macro_ap > progress['best_macro_ap']
    )

    # Retain models using validation Macro AP only

    if improved:

        best_model_path = save_best_model(
            model,
            S02_MODEL_DIR,
            epoch
        )

        checkpoint_manifest = read_json(
            best_model_path / 'checkpoint_manifest.json'
        )

        verify_model_files(
            best_model_path,
            checkpoint_manifest
        )

        best_validation_path = (
            S02_PREDICTIONS_DIR
            / f'{best_model_path.name}_validation.npz'
        )

        temporary_validation_path = (
            best_validation_path.with_suffix('.tmp.npz')
        )

        np.savez_compressed(
            temporary_validation_path,
            ids=np.asarray(val_comment_ids, dtype=str),
            targets=y_val,
            logits=validation_result['logits'],
            scores=validation_result['scores']
        )

        with np.load(
            temporary_validation_path,
            allow_pickle=False
        ) as saved:

            np.testing.assert_array_equal(
                saved['ids'],
                val_comment_ids
            )

            np.testing.assert_array_equal(
                saved['targets'],
                y_val
            )

            np.testing.assert_array_equal(
                saved['scores'],
                validation_result['scores']
            )

        temporary_validation_path.replace(
            best_validation_path
        )

        progress['best_macro_ap'] = macro_ap
        progress['best_epoch'] = epoch

        progress['best_checkpoint_path'] = str(
            best_model_path.relative_to(PROJECT_ROOT)
        )

        progress['best_validation_path'] = str(
            best_validation_path.relative_to(PROJECT_ROOT)
        )

        progress['best_validation_sha256'] = (
            calculate_sha256(best_validation_path)
        )

        progress['epochs_without_improvement'] = 0

        print("New best validation model saved.")

    else:

        progress['epochs_without_improvement'] += 1

    # Record epoch-level evidence

    epoch_record = {
        'Epoch': epoch,
        'Training Loss': float(training_loss),
        'Validation Loss': float(
            validation_result['validation_loss']
        ),
        'Validation Macro AP': macro_ap,
        'Validation Macro ROC-AUC': macro_roc,
        'Validation Macro F1': macro_f1,
        'Validation Micro F1': micro_f1,
        'Attempted Updates': progress['attempted_updates'],
        'Successful Updates': progress['successful_updates'],
        'Skipped Updates': progress['skipped_updates'],
        'Learning Rate': float(
            optimizer.param_groups[0]['lr']
        ),
        'Cumulative Training Time Seconds': float(
            progress['training_time_seconds']
        ),
        'Validation Inference Seconds': float(
            validation_result['inference_time_seconds']
        ),
        'Peak GPU Allocated GiB': (
            torch.cuda.max_memory_allocated() / 1024**3
        ),
        'Peak GPU Reserved GiB': (
            torch.cuda.max_memory_reserved() / 1024**3
        ),
        'Best Model Improved': bool(improved)
    }

    progress['history'].append(epoch_record)

    print("\nEpoch results")

    display(
        pd.DataFrame([epoch_record]).round(6)
    )

    # Prepare the next epoch or finish training

    if epoch >= MAX_EPOCHS:

        progress['training_finished'] = True
        progress['stopping_reason'] = 'maximum_epochs'

    elif (
        progress['epochs_without_improvement']
        >= EARLY_STOPPING_PATIENCE
    ):

        progress['training_finished'] = True
        progress['stopping_reason'] = 'early_stopping'

    else:

        progress['epoch'] += 1
        progress['next_row'] = 0
        progress['epoch_examples'] = 0
        progress['epoch_loss_sum'] = 0.0

    # Save recovery before removing superseded models

    persist_s02_recovery(progress)

    if improved and previous_best_path:

        old_model_path = PROJECT_ROOT / previous_best_path

        assert old_model_path.resolve().is_relative_to(
            S02_MODEL_DIR.resolve()
        )

        assert previous_best_path != (
            progress['best_checkpoint_path']
        )

        if old_model_path.exists():

            shutil.rmtree(old_model_path)

            print("Superseded best checkpoint removed.")

    if improved and previous_best_predictions:

        old_predictions = (
            PROJECT_ROOT / previous_best_predictions
        )

        assert old_predictions.resolve().is_relative_to(
            S02_PREDICTIONS_DIR.resolve()
        )

        assert previous_best_predictions != (
            progress['best_validation_path']
        )

        old_predictions.unlink(missing_ok=True)

    del validation_result

    gc.collect()
    torch.cuda.empty_cache()


# Verify completed training

assert progress['training_finished']

assert progress['best_epoch'] is not None

assert progress['attempted_updates'] == (
    len(progress['history']) * UPDATES_PER_EPOCH
)

assert (
    progress['successful_updates']
    + progress['skipped_updates']
    == progress['attempted_updates']
)

S02_BEST_MODEL_PATH = (
    PROJECT_ROOT / progress['best_checkpoint_path']
)

S02_BEST_VALIDATION_PATH = (
    PROJECT_ROOT / progress['best_validation_path']
)

best_manifest = read_json(
    S02_BEST_MODEL_PATH / 'checkpoint_manifest.json'
)

verify_model_files(
    S02_BEST_MODEL_PATH,
    best_manifest
)

assert calculate_sha256(
    S02_BEST_VALIDATION_PATH
) == progress['best_validation_sha256']

assert progress['best_macro_ap'] == max(
    record['Validation Macro AP']
    for record in progress['history']
)

S02_TRAINING_SUMMARY_PATH = (
    S02_LOGS_DIR / 'training_completion_summary.json'
)

training_summary = {
    'run_id': RUN_ID,
    'reference_run_id': 'T06_FOCAL_S42',
    'training_samples': TRAINING_SAMPLES,
    'validation_samples': VALIDATION_SAMPLES,
    'epochs_completed': len(progress['history']),
    'best_epoch': progress['best_epoch'],
    'best_validation_macro_ap': progress['best_macro_ap'],
    'attempted_updates': progress['attempted_updates'],
    'successful_updates': progress['successful_updates'],
    'skipped_updates': progress['skipped_updates'],
    'training_time_seconds': progress['training_time_seconds'],
    'current_session_wall_time_seconds': (
        time.perf_counter() - session_start
    ),
    'best_model_path': progress['best_checkpoint_path'],
    'best_model_state_sha256': (
        best_manifest['model_state_sha256']
    ),
    'best_validation_path': progress['best_validation_path'],
    'best_validation_sha256': progress['best_validation_sha256'],
    'initial_model_state_sha256': INITIAL_STATE_HASH,
    'stopping_reason': progress['stopping_reason'],
    'training_completed': True,
    'final_metrics_verified': False,
    'experiment_registered': False,
    'official_test_used': False,
    'final_project_model_frozen': False
}

save_json_atomic(
    S02_TRAINING_SUMMARY_PATH,
    training_summary
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_hash_before
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_hash_before
)

print("\nFinal epoch history")

display(
    pd.DataFrame(progress['history']).round(6)
)

print("\nS02 training completion summary")

for key, value in training_summary.items():

    print(f"{key}: {value}")

print("\nBest validation model: Saved and verified")
print("Best validation predictions: Saved")
print("Recovery checkpoint: Retained until final verification")
print("Experiment registry: Unchanged")
print("Official test dataset: Not loaded")

del model, optimizer, scheduler, scaler

gc.collect()
torch.cuda.empty_cache()

print("\nCell 7 completed successfully.")

In [ ]:
script_path = SRC_DIR / '06_optimization_cell8.py'
exec(compile(script_path.read_text(), str(script_path), 'exec'), globals())


In [ ]:
script_path = SRC_DIR / '06_optimization_cell9.py'
exec(compile(script_path.read_text(), str(script_path), 'exec'), globals())


In [ ]:
script_path = SRC_DIR / '06_optimization_cell10.py'
exec(compile(script_path.read_text(), str(script_path), 'exec'), globals())
